# Phase 9 v2 — Hybrid RAG + Deterministic Tool Use + Grounded Answer Repair

## Qwen2.5 + Phase 1/Phase 2 merge + FAISS + BM25 + RRF + Cross-Encoder + Metadata + LangChain

Phiên bản này được nâng cấp trực tiếp từ output thực tế của Phase 9 trước.

### Đánh giá phiên chạy trước

Điểm tốt:

- Corpus enriched load đủ **58.603/58.603** chunk.
- FAISS + BM25 cache hoạt động.
- OOD smoke test chặn đúng câu hỏi Python.
- 10/10 manual history questions đều chạy tới `STATUS: ok`.
- `support_score` 10 câu manual khoảng **0.854–0.907**, trung bình khoảng **0.880**.
- Latency manual khoảng **28.5–58.4 giây/câu**, trung bình khoảng **42 giây/câu**.
- Retrieval nhìn chung lấy đúng chủ đề, năm và thực thể lịch sử.

Vấn đề thấy rõ:

1. **10/10 câu manual mở đầu bằng “Theo tài liệu”**.
2. Có câu dùng từ **“Chunk cũng nêu...”**.
3. Một số câu chưa trả đủ ý:
   - Lam Sơn hỏi *bối cảnh + kết quả* nhưng chủ yếu kể bối cảnh.
   - So sánh nhà Lý / nhà Trần chưa thực sự đối chiếu và có dấu hiệu trộn thông tin.
   - Ba lần kháng chiến Mông–Nguyên hỏi *đặc điểm + ý nghĩa* nhưng phần ý nghĩa gần như thiếu.
   - Tết Mậu Thân hỏi *phe nào thắng* nhưng model né kết luận.
4. Context cuối đôi khi có quá nhiều chunk cùng một title.
5. `STATUS: ok` mới kiểm tra source/year, chưa kiểm tra answer completeness/directness.
6. Benchmark 4 cấu hình chưa hoàn tất vì `KeyboardInterrupt` ở vanilla.

---

## Pipeline v2

```text
Question
  ↓
[Tool 1] Question Analyzer
  ↓
[Tool 2] Query Planner
  ↓
FAISS + BM25 cho từng query
  ↓
Weighted multi-query RRF
  ↓
Top 20 candidates
  ↓
Cross-Encoder reranker
  ↓
Metadata intent-aware soft boost
  ↓
[Tool 3] Evidence Coverage + Title Diversity
  ↓
Top context
  ↓
Qwen Phase1 + Phase2
  ↓
[Tool 4] Style Polisher
  ↓
Source / year guards
  ↓
[Tool 5] Answer Critic
  ↓
nếu cần: 1 lần evidence-only repair
  ↓
Final grounded answer
```

### Vì sao không để Qwen-3B tự quyết định gọi tool?

Với mục tiêu chống bịa, deterministic orchestration ổn định hơn agentic function-calling trên model 3B:

- LangChain điều phối tool theo pipeline cố định.
- LLM chỉ tổng hợp/repair câu trả lời.
- Retrieval, metadata, validation và critic không phụ thuộc model có nhớ gọi tool hay không.


## Nâng cấp chính trong v2

- Bỏ boilerplate **“Theo tài liệu...”** ở prompt và post-processing.
- Không cho answer user-facing dùng từ **“chunk”**.
- Dynamic rules theo loại câu hỏi: `winner`, `compare`, `context`, `outcome`, `significance`, `content`...
- Multi-query retrieval theo intent nhưng vẫn giữ **RRF Top 20**.
- Final context mặc định tối đa **2 chunk / title**.
- Metadata `content_facets` được boost theo intent, kể cả khi cách diễn đạt câu hỏi khác metadata.
- Answer critic kiểm tra câu trả lời có trực tiếp và đủ ý hay không.
- Nếu thiếu ý hoặc vi phạm source/year guard, cho Qwen **repair đúng 1 lần bằng cùng evidence**.
- Benchmark v2 dùng checkpoint riêng, không trộn output cũ.


## 1. Cài dependencies

Cài các thư viện cần cho inference và evaluation.

**Chạy một lần ở đầu runtime.** Nếu Colab yêu cầu restart sau `pip install`, restart rồi tiếp tục từ Cell 2.

**GPU:** chưa bắt buộc ở cell này.


In [1]:
# Cell 1 — Install dependencies
# Chạy đầu tiên. Nếu torch/transformers đã import trước cell này, restart runtime sau khi cài.

%pip -q install -U \
  "transformers>=4.51,<5" \
  "peft>=0.13,<1" \
  "accelerate>=1.0,<2" \
  "sentence-transformers>=3.4,<6" \
  "faiss-cpu>=1.8" \
  "bm25s>=0.2.14" \
  "langchain-core>=0.3,<2" \
  "scikit-learn>=1.4" \
  "pandas>=2.0" \
  "safetensors>=0.4" \
  "tqdm>=4.66" \
  "torchao==0.17.0"

print("✅ Dependencies installed.")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 8.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 87.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 154.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 58.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 596.7/596.7 kB 50.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 132.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.7/74.7 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.7/561.7 kB 55.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 155.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 152.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

## 2. Mount Google Drive và cấu hình toàn hệ thống

Cell này:

- mount Google Drive;
- import thư viện;
- khai báo đường dẫn corpus / adapter / cache;
- cấu hình FAISS, BM25, reranker, generation và benchmark;
- kiểm tra CUDA/GPU hiện tại.

### GPU khuyến nghị

- **A100 40/80 GB:** lựa chọn tốt nhất.
- **L4 24 GB:** phù hợp nếu chỉ inference hoặc benchmark chậm hơn.
- **T4 16 GB:** có thể chạy nhưng cần theo dõi VRAM.

Nếu muốn thay model, batch size hoặc số context cuối, chỉnh chủ yếu tại cell này.


In [2]:
# Cell 2 — Mount Drive, imports, global config

from google.colab import drive
drive.mount('/content/drive')

import os, re, gc, json, glob, time, math, shutil, zipfile, hashlib, random, unicodedata
from pathlib import Path
from typing import Any, Dict, List, Tuple, Optional
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import torch
import faiss
import bm25s

from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from sklearn.model_selection import train_test_split
from langchain_core.runnables import RunnableLambda
from langchain_core.tools import tool
from IPython.display import display

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

PIPELINE_VERSION = "phase9_v2_tooluse_grounded_direct"

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
DRIVE_ROOT = Path('/content/drive/MyDrive')
BACKUP_DIR = DRIVE_ROOT / 'vn_history_model_backups'
RAG_ROOT = BACKUP_DIR / 'rag_corpus_vn_history'
PHASE8_METADATA_DIR = RAG_ROOT / 'metadata'
PHASE9_DIR = RAG_ROOT / 'phase9_hybrid_rag'
PHASE9_DIR.mkdir(parents=True, exist_ok=True)

ENRICHED_CANDIDATES = [
    PHASE8_METADATA_DIR / 'vn_history_rag_chunks_enriched.jsonl',
    RAG_ROOT / 'processed' / 'vn_history_rag_chunks_enriched.jsonl',
]
RAW_CORPUS_PATH = RAG_ROOT / 'processed' / 'vn_history_rag_chunks.jsonl'
METADATA_PATH = PHASE8_METADATA_DIR / 'vn_history_rag_chunk_metadata.jsonl'

SFT_DATA_DIR = DRIVE_ROOT / 'vn_history_rag_sft_dataset'
MESSAGES_PATH = SFT_DATA_DIR / 'all_messages.jsonl'

PHASE1_ADAPTER_CANDIDATES = [
    BACKUP_DIR / 'qwen_vnhistory_phase1_best_adapter',
    BACKUP_DIR / 'qwen_vnhistory_phase1_best_adapter.zip',
]
PHASE2_ADAPTER_CANDIDATES = [
    BACKUP_DIR / 'qwen_vnhistory_phase6_rag_best_adapter',
    BACKUP_DIR / 'qwen_vnhistory_phase6_rag_best_adapter.zip',
    BACKUP_DIR / 'qwen2_5_3b_vnhistory_phase6_rag_qlora_best_by_generation_metric',
    BACKUP_DIR / 'qwen2_5_3b_vnhistory_phase6_rag_qlora_best_by_generation_metric.zip',
]

EMBEDDING_MODEL_ID = 'intfloat/multilingual-e5-base'
RERANKER_MODEL_ID = 'BAAI/bge-reranker-v2-m3'

EMBED_BATCH_SIZE = 128
DENSE_FETCH_K = 80
BM25_FETCH_K = 80
RRF_K = 60
RRF_TOP_K = 20
FINAL_CONTEXT_K = 6
RERANK_BATCH_SIZE = 32

MAX_QUERY_VARIANTS = 3
QUERY_EXPANSION_WEIGHT = 0.82
MAX_CHUNKS_PER_TITLE = 2
ENABLE_CONTEXT_DIVERSITY = True

FORCE_REBUILD_FAISS = False
FORCE_REBUILD_BM25 = False

METADATA_MAX_BONUS = 0.18
INTENT_FACET_BONUS = 0.025

MAX_INPUT_TOKENS = 3600
MAX_NEW_TOKENS = 300
MAX_CHARS_PER_CHUNK = 1800
MIN_CHARS_PER_CHUNK = 550

TEMPERATURE = 0.0
TOP_P = 1.0
REPETITION_PENALTY = 1.05

STRICT_SOURCE_REQUIRED = True
STRICT_UNSUPPORTED_YEAR_GUARD = True

ENABLE_COMPLETENESS_REWRITE = True
MAX_REWRITE_ATTEMPTS = 1
SHOW_TOOL_TRACE = True

OOD_ANCHOR_MARGIN = 0.02
SECONDARY_OOD_MARGIN = -0.06
SECONDARY_MIN_DENSE = 0.28

BENCHMARK_HISTORY_N = 90
BENCHMARK_OOD_N = 10

GENERATION_BATCH_SIZE = 16
BENCHMARK_MAX_NEW_TOKENS = 180
BENCHMARK_SAVE_EVERY = 5
BENCHMARK_PATH = PHASE9_DIR / 'benchmark_100.jsonl'
BENCHMARK_RESULTS_PATH = PHASE9_DIR / 'benchmark_results_v2_tooluse.jsonl'
BENCHMARK_SUMMARY_PATH = PHASE9_DIR / 'benchmark_summary_v2_tooluse.csv'

CUDA_AVAILABLE = torch.cuda.is_available()
GPU_VRAM_GB = 0.0

print('CUDA:', CUDA_AVAILABLE)
if CUDA_AVAILABLE:
    print('GPU:', torch.cuda.get_device_name(0))
    GPU_VRAM_GB = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f'GPU VRAM: {GPU_VRAM_GB:.1f} GB')

print('PHASE9_DIR:', PHASE9_DIR)
print('PIPELINE_VERSION:', PIPELINE_VERSION)

if not CUDA_AVAILABLE:
    print('⚠️ Không có CUDA GPU — full inference/benchmark sẽ rất chậm.')
elif GPU_VRAM_GB >= 38:
    print('✅ GPU recommendation: EXCELLENT — A100-class.')
elif GPU_VRAM_GB >= 22:
    print('✅ GPU recommendation: GOOD — L4/24GB-class.')
elif GPU_VRAM_GB >= 15:
    print('⚠️ GPU recommendation: TIGHT — T4-class, theo dõi OOM.')
else:
    print('⚠️ GPU recommendation: LOW VRAM.')


Mounted at /content/drive


CUDA: True
GPU: NVIDIA A100-SXM4-80GB
GPU VRAM: 79.3 GB
PHASE9_DIR: /content/drive/MyDrive/vn_history_model_backups/rag_corpus_vn_history/phase9_hybrid_rag
PIPELINE_VERSION: phase9_v2_tooluse_grounded_direct
✅ GPU recommendation: EXCELLENT — A100-class.


## 3. Utilities và tự động tìm file

Định nghĩa các helper dùng xuyên suốt notebook:

- đọc/ghi JSONL;
- normalize text;
- hash/signature corpus;
- tự tìm enriched corpus;
- tự tìm và giải nén adapter Phase 1 / Phase 2;
- cleanup CUDA.

Cell này không thực hiện inference nên chạy khá nhẹ.


In [3]:
# Cell 3 — Utilities: JSONL, normalization, corpus/adapters auto-resolve

def read_jsonl(path: Path) -> List[Dict[str, Any]]:
    rows = []
    with Path(path).open('r', encoding='utf-8') as f:
        for line_no, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError as e:
                raise ValueError(f'JSON lỗi {path} dòng {line_no}: {e}') from e
    return rows


def append_jsonl(path: Path, rows: List[Dict[str, Any]]) -> None:
    if not rows:
        return
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('a', encoding='utf-8') as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + '\n')
        f.flush()


def write_json(path: Path, obj: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w', encoding='utf-8') as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)


def clean_text(s: Any) -> str:
    s = '' if s is None else str(s)
    s = s.replace('\r\n', '\n').replace('\r', '\n')
    s = re.sub(r'[ \t]+', ' ', s)
    s = re.sub(r'\n{3,}', '\n\n', s)
    return s.strip()


def strip_accents(s: str) -> str:
    s = unicodedata.normalize('NFD', clean_text(s).lower())
    s = ''.join(ch for ch in s if unicodedata.category(ch) != 'Mn')
    return s.replace('đ', 'd')


def match_norm(s: str) -> str:
    s = strip_accents(s)
    s = re.sub(r'[^a-z0-9]+', ' ', s)
    return re.sub(r'\s+', ' ', s).strip()


def short_text(s: str, max_chars: int) -> str:
    s = clean_text(s)
    if len(s) <= max_chars:
        return s
    cut = s[:max_chars]
    last = max(cut.rfind('. '), cut.rfind('; '), cut.rfind('\n'), cut.rfind(' '))
    if last > max_chars * 0.65:
        cut = cut[:last]
    return cut.strip() + ' ...'


def corpus_signature(rows: List[Dict[str, Any]]) -> str:
    h = hashlib.sha1()
    for r in rows:
        cid = str(r.get('chunk_id', ''))
        text_hash = str(r.get('text_hash', ''))
        if not text_hash:
            text_hash = hashlib.sha1((clean_text(r.get('title')) + '\n' + clean_text(r.get('text'))).encode('utf-8')).hexdigest()[:16]
        h.update(f'{cid}|{text_hash}\n'.encode('utf-8'))
    return h.hexdigest()[:20]


def resolve_enriched_corpus() -> Path:
    for p in ENRICHED_CANDIDATES:
        if p.exists() and p.stat().st_size > 0:
            return p

    # Fallback: join raw corpus + metadata bằng chunk_id nếu enriched chưa có.
    if RAW_CORPUS_PATH.exists() and METADATA_PATH.exists():
        print('Không thấy enriched file; sẽ join raw + metadata trong RAM.')
        return Path('__JOIN_RAW_METADATA__')

    raise FileNotFoundError(
        'Không tìm thấy enriched corpus, cũng không đủ raw+metadata để join.\n' +
        '\n'.join(map(str, ENRICHED_CANDIDATES + [RAW_CORPUS_PATH, METADATA_PATH]))
    )


def resolve_adapter_dir(name: str, candidates: List[Path]) -> str:
    """Giống Phase 7: nhận folder trực tiếp hoặc zip, tự extract nếu cần."""
    extract_root = Path('/content/adapters') / name
    extract_root.parent.mkdir(parents=True, exist_ok=True)

    for p in candidates:
        p = Path(p)
        if p.is_dir() and (p / 'adapter_config.json').exists():
            return str(p)
        if p.is_file() and p.suffix.lower() == '.zip':
            if (extract_root / 'adapter_config.json').exists():
                return str(extract_root)
            print(f'Extracting {name} adapter:', p)
            if extract_root.exists():
                shutil.rmtree(extract_root)
            extract_root.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(p, 'r') as z:
                z.extractall(extract_root)
            matches = list(extract_root.rglob('adapter_config.json'))
            if not matches:
                raise FileNotFoundError(f'Không thấy adapter_config.json trong {p}')
            return str(matches[0].parent)

    # fallback recursive search trong backup dir
    matches = list(BACKUP_DIR.rglob('adapter_config.json'))
    for m in matches:
        if name.lower() in str(m).lower():
            return str(m.parent)

    raise FileNotFoundError(f'Không tìm thấy adapter {name}. Candidates: {candidates}')


def get_model_device(model):
    try:
        return model.get_input_embeddings().weight.device
    except Exception:
        return next(model.parameters()).device


def cleanup_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

## 4. Load enriched corpus từ Phase 8

Notebook ưu tiên:

```text
vn_history_rag_chunks_enriched.jsonl
```

Nếu không tìm thấy, notebook có thể ghép raw corpus với metadata bằng `chunk_id`.

Sau khi load, cell kiểm tra:

- số lượng chunk;
- `chunk_id`;
- duplicate;
- metadata;
- corpus signature.

> Corpus enriched là **source chính cho retrieval**. Metadata chỉ dùng để soft-boost, không thay thế raw text.


In [4]:
# Cell 4 — Load enriched corpus (auto) + strict structural checks

resolved = resolve_enriched_corpus()

if str(resolved) == '__JOIN_RAW_METADATA__':
    raw_rows = read_jsonl(RAW_CORPUS_PATH)
    metadata_rows = read_jsonl(METADATA_PATH)
    metadata_by_id = {str(m['chunk_id']): m for m in metadata_rows}
    chunks = []
    for c in raw_rows:
        cid = str(c['chunk_id'])
        e = dict(c)
        e['metadata'] = metadata_by_id.get(cid, {})
        chunks.append(e)
    CORPUS_PATH = RAW_CORPUS_PATH
else:
    CORPUS_PATH = resolved
    chunks = read_jsonl(CORPUS_PATH)

if not chunks:
    raise RuntimeError('Corpus rỗng.')

required = {'chunk_id', 'title', 'text'}
for i, c in enumerate(chunks[:50]):
    miss = required - set(c)
    if miss:
        raise ValueError(f'Chunk {i} thiếu {sorted(miss)}')

chunk_ids = [str(c['chunk_id']) for c in chunks]
if len(set(chunk_ids)) != len(chunk_ids):
    raise ValueError('Corpus có chunk_id trùng.')

chunk_by_id = {str(c['chunk_id']): c for c in chunks}
CORPUS_ID_SET = set(chunk_by_id)
CORPUS_SIGNATURE = corpus_signature(chunks)

metadata_nonempty = sum(bool(c.get('metadata')) for c in chunks)
print('CORPUS_PATH:', CORPUS_PATH)
print('Chunks:', f'{len(chunks):,}')
print('Metadata present:', f'{metadata_nonempty:,}/{len(chunks):,}')
print('Corpus signature:', CORPUS_SIGNATURE)
print('Sample keys:', sorted(chunks[0].keys()))

CORPUS_PATH: /content/drive/MyDrive/vn_history_model_backups/rag_corpus_vn_history/metadata/vn_history_rag_chunks_enriched.jsonl
Chunks: 58,603
Metadata present: 58,603/58,603
Corpus signature: ef60222b683077c6b72c
Sample keys: ['char_len', 'chunk_filter_reason', 'chunk_history_score', 'chunk_id', 'chunk_index', 'doc_filter_reason', 'filter_version', 'hf_dataset', 'history_score', 'matched_vn_history_terms', 'metadata', 'raw_record_index', 'section', 'source', 'source_type', 'text', 'text_hash', 'title', 'url', 'word_len']


## 5. Dense retrieval — multilingual E5 + FAISS

Tạo embedding cho `title + text` và build FAISS `IndexFlatIP`.

- Query dùng prefix `query:`.
- Passage dùng prefix `passage:`.
- Embedding được normalize để cosine similarity tương đương inner product.
- Cache được lưu trên Drive và chỉ rebuild khi corpus/model thay đổi.

### GPU

- **A100/L4 khuyến nghị** ở lần build đầu.
- Nếu FAISS cache đã tồn tại, cell chủ yếu chỉ load index nên nhanh hơn nhiều.
- T4 có thể dùng nhưng nên giảm `EMBED_BATCH_SIZE` nếu OOM.


In [5]:
# Cell 5 — Build/load E5 FAISS index trên toàn bộ enriched corpus

safe_embed = re.sub(r'[^A-Za-z0-9._-]+', '_', EMBEDDING_MODEL_ID)
FAISS_DIR = PHASE9_DIR / f'faiss_{safe_embed}'
FAISS_DIR.mkdir(parents=True, exist_ok=True)
FAISS_INDEX_PATH = FAISS_DIR / 'chunks.index'
FAISS_MANIFEST_PATH = FAISS_DIR / 'manifest.json'

embed_device = 'cuda' if torch.cuda.is_available() else 'cpu'
embedder = SentenceTransformer(EMBEDDING_MODEL_ID, device=embed_device)
try:
    embedder.max_seq_length = 512
except Exception:
    pass


def passage_for_embedding(c: Dict[str, Any]) -> str:
    return 'passage: ' + clean_text(f"{c.get('title','')}\n{c.get('text','')}")


def query_for_embedding(q: str) -> str:
    return 'query: ' + clean_text(q)


def faiss_cache_valid() -> bool:
    if FORCE_REBUILD_FAISS or not FAISS_INDEX_PATH.exists() or not FAISS_MANIFEST_PATH.exists():
        return False
    try:
        manifest = json.loads(FAISS_MANIFEST_PATH.read_text(encoding='utf-8'))
        return (
            manifest.get('corpus_signature') == CORPUS_SIGNATURE
            and manifest.get('embedding_model') == EMBEDDING_MODEL_ID
            and int(manifest.get('count', -1)) == len(chunks)
        )
    except Exception:
        return False


if faiss_cache_valid():
    print('Loading FAISS cache:', FAISS_INDEX_PATH)
    faiss_index = faiss.read_index(str(FAISS_INDEX_PATH))
else:
    print('Building FAISS embeddings for', f'{len(chunks):,}', 'chunks...')
    texts = [passage_for_embedding(c) for c in chunks]
    embeddings = embedder.encode(
        texts,
        batch_size=EMBED_BATCH_SIZE,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    ).astype('float32')

    faiss_index = faiss.IndexFlatIP(embeddings.shape[1])
    faiss_index.add(embeddings)

    local_idx = Path('/content/phase9_chunks.index')
    faiss.write_index(faiss_index, str(local_idx))
    shutil.copy2(local_idx, FAISS_INDEX_PATH)

    write_json(FAISS_MANIFEST_PATH, {
        'corpus_signature': CORPUS_SIGNATURE,
        'embedding_model': EMBEDDING_MODEL_ID,
        'count': len(chunks),
        'dim': int(embeddings.shape[1]),
    })
    del embeddings, texts
    cleanup_cuda()

assert faiss_index.ntotal == len(chunks), (faiss_index.ntotal, len(chunks))
print('FAISS ready:', faiss_index.ntotal)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Loading FAISS cache: /content/drive/MyDrive/vn_history_model_backups/rag_corpus_vn_history/phase9_hybrid_rag/faiss_intfloat_multilingual-e5-base/chunks.index
FAISS ready: 58603


## 6. Lexical retrieval — BM25S

Build hoặc load BM25 index trên toàn bộ corpus.

BM25 bổ sung cho FAISS trong các trường hợp:

- tên riêng;
- niên đại;
- cụm từ lịch sử đặc thù;
- từ khóa hiếm.

BM25 thiên về CPU/RAM và không cần GPU mạnh. Cache cũng được giữ lại cho những lần chạy sau.


In [6]:
# Cell 6 — Build/load BM25S index (memory-efficient BM25)

BM25_DIR = PHASE9_DIR / 'bm25s_index'
BM25_MANIFEST_PATH = BM25_DIR / 'phase9_manifest.json'


def bm25_text(c: Dict[str, Any]) -> str:
    # title lặp 2 lần để lexical retrieval ưu tiên entity/title nhưng vẫn giữ full text.
    title = clean_text(c.get('title', ''))
    text = clean_text(c.get('text', ''))
    return match_norm(f'{title} {title} {text}')


def bm25_cache_valid() -> bool:
    if FORCE_REBUILD_BM25 or not BM25_DIR.exists() or not BM25_MANIFEST_PATH.exists():
        return False
    try:
        manifest = json.loads(BM25_MANIFEST_PATH.read_text(encoding='utf-8'))
        return (
            manifest.get('corpus_signature') == CORPUS_SIGNATURE
            and int(manifest.get('count', -1)) == len(chunks)
        )
    except Exception:
        return False


if bm25_cache_valid():
    print('Loading BM25S mmap cache:', BM25_DIR)
    bm25_retriever = bm25s.BM25.load(str(BM25_DIR), mmap=True, load_corpus=False)
else:
    print('Building BM25S index...')
    BM25_DIR.mkdir(parents=True, exist_ok=True)
    lexical_corpus = [bm25_text(c) for c in tqdm(chunks, desc='Prepare BM25 text')]
    corpus_tokens = bm25s.tokenize(lexical_corpus, stopwords=None, stemmer=None)
    bm25_retriever = bm25s.BM25()
    bm25_retriever.index(corpus_tokens)
    bm25_retriever.save(str(BM25_DIR))
    write_json(BM25_MANIFEST_PATH, {
        'corpus_signature': CORPUS_SIGNATURE,
        'count': len(chunks),
        'note': 'title duplicated x2 + full text, normalized for Vietnamese lexical retrieval',
    })
    del lexical_corpus, corpus_tokens
    gc.collect()

print('BM25 ready.')

Loading BM25S mmap cache: /content/drive/MyDrive/vn_history_model_backups/rag_corpus_vn_history/phase9_hybrid_rag/bm25s_index
BM25 ready.


## 7. Cross-Encoder + OOD + deterministic retrieval tools

Khởi tạo:

- `BAAI/bge-reranker-v2-m3`;
- history anchors;
- off-topic anchors;
- rule phát hiện câu hỏi lạc đề;
- hàm tính metadata bonus.

Metadata chỉ được dùng như **soft bonus**. Chunk thiếu metadata vẫn có thể được retrieval bình thường.

**GPU:** A100/L4 tốt nhất; T4 chạy được nhưng rerank sẽ chậm hơn.


In [7]:
# Cell 7 — Reranker + deterministic tool layer

rerank_device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Loading reranker:', RERANKER_MODEL_ID)
reranker = CrossEncoder(RERANKER_MODEL_ID, max_length=512, device=rerank_device)

HISTORY_ANCHORS = [
    'lịch sử Việt Nam các triều đại và nhà nước',
    'khởi nghĩa kháng chiến chiến tranh trong lịch sử Việt Nam',
    'nhân vật lịch sử Việt Nam vua tướng lãnh tụ',
    'hiệp định ngoại giao cách mạng Việt Nam',
    'nhà Lý nhà Trần nhà Lê nhà Nguyễn',
    'Cách mạng tháng Tám chiến tranh Việt Nam',
]
OOD_ANCHORS = [
    'lập trình Python JavaScript sửa lỗi phần mềm',
    'thời tiết hôm nay nhiệt độ dự báo mưa',
    'bóng đá cầu thủ câu lạc bộ tỉ số trận đấu',
    'nấu ăn công thức món ăn nguyên liệu',
    'toán học phương trình tính toán',
    'triệu chứng bệnh thuốc điều trị y tế',
    'giá cổ phiếu tiền điện tử tài chính hôm nay',
    'tình yêu quan hệ cá nhân hẹn hò',
]

anchor_texts = ['query: ' + x for x in HISTORY_ANCHORS + OOD_ANCHORS]
anchor_embs = embedder.encode(anchor_texts, convert_to_numpy=True, normalize_embeddings=True).astype('float32')
HISTORY_ANCHOR_EMBS = anchor_embs[:len(HISTORY_ANCHORS)]
OOD_ANCHOR_EMBS = anchor_embs[len(HISTORY_ANCHORS):]

EXPLICIT_OOD_PATTERNS = [
    r'\bpython\b|\bjavascript\b|\bjava\b|lap trinh|viet code|debug|\bapi\b',
    r'thoi tiet|nhiet do hom nay|du bao mua|do am hom nay',
    r'\bmessi\b|\bronaldo\b|premier league|champions league|ti so bong da|world cup 20\d\d',
    r'cong thuc nau|nau mon|chien bao lau|luoc bao lau',
    r'giai phuong trinh|dao ham|tich phan|tinh \d+\s*[+*/-]',
    r'trieu chung|lieu thuoc|uong thuoc|dau bung|sot bao nhieu',
    r'gia bitcoin|gia co phieu|ty gia hom nay|mua coin',
    r'iphone|samsung|dien thoai nao',
    r'dich cau|dich sang tieng viet|translate',
]

FACET_PATTERNS = [
    ('winner',       r'phe nao.*thang|ben nao.*thang|ai thang|chien thang cua phe|thang hay thua|thang loi cua ai'),
    ('compare',      r'\bso sanh\b|khac nhau|giong nhau|doi chieu'),
    ('context',      r'boi canh|hoan canh'),
    ('cause',        r'nguyen nhan|vi sao|tai sao|do dau'),
    ('outcome',      r'ket qua|hau qua|ket thuc|ra sao'),
    ('significance', r'y nghia|vai tro|tac dong'),
    ('process',      r'dien ra nhu the nao|dien bien|qua trinh'),
    ('content',      r'noi dung|dieu khoan|quy dinh'),
    ('features',     r'dac diem|noi bat'),
    ('time',         r'khi nao|thoi gian nao|vao nam nao|nam nao'),
]

FACET_QUERY_SUFFIX = {
    'winner': 'kết quả quân sự chiến thuật chiến lược chính trị bên thắng bên thua tổn thất mục tiêu',
    'compare': 'so sánh vai trò điểm giống khác đóng góp xây dựng bảo vệ',
    'context': 'bối cảnh hoàn cảnh trước khi nguyên nhân dẫn tới',
    'cause': 'nguyên nhân lý do điều kiện dẫn đến',
    'outcome': 'kết quả kết thúc thắng lợi thất bại hậu quả hệ quả',
    'significance': 'ý nghĩa tác động vai trò đánh dấu mở ra góp phần',
    'process': 'diễn biến quá trình các giai đoạn mốc chính',
    'content': 'nội dung điều khoản quy định chính',
    'features': 'đặc điểm nổi bật tính chất lực lượng hình thức',
    'time': 'thời gian niên đại mốc năm',
}

FACET_COVERAGE_TERMS = {
    'winner': ['chien thang','thang loi','that bai','uu the','quan su','chien luoc','chinh tri','ton that'],
    'context': ['boi canh','hoan canh','truoc khi','sau khi','xam luoc','do ','vi ','khi '],
    'cause': ['nguyen nhan','do ','vi ','dan den','nguyen do'],
    'outcome': ['ket qua','ket thuc','thang loi','that bai','cham dut','gianh','buoc','thanh lap','thoai vi','tao co so'],
    'significance': ['y nghia','danh dau','khang dinh','mo ra','gop phan','tac dong','vai tro','tao tien de'],
    'process': ['dien bien','qua trinh','tien cong','khoi nghia','chien dich','giai doan'],
    'content': ['noi dung','dieu khoan','quy dinh','ngung ban','hiep dinh','cam ket'],
    'features': ['dac diem','noi bat','tinh chat','luc luong'],
}
FACET_TO_METADATA = {
    'context': {'boi canh'},
    'cause': {'nguyen nhan'},
    'outcome': {'ket qua'},
    'winner': {'ket qua'},
    'significance': {'y nghia'},
    'process': {'dien bien'},
    'content': {'noi dung'},
    'features': {'dac diem'},
}

def intent_scores(question: str) -> Dict[str, Any]:
    q_emb = embedder.encode([query_for_embedding(question)], convert_to_numpy=True, normalize_embeddings=True).astype('float32')[0]
    hs = float(np.max(HISTORY_ANCHOR_EMBS @ q_emb))
    oscore = float(np.max(OOD_ANCHOR_EMBS @ q_emb))
    qn = match_norm(question)
    explicit = any(re.search(p, qn, flags=re.I) for p in EXPLICIT_OOD_PATTERNS)
    return {'history_anchor':hs,'ood_anchor':oscore,'margin':hs-oscore,'explicit_ood':bool(explicit),'query_embedding':q_emb}

def analyze_question_core(question: str) -> Dict[str, Any]:
    question = clean_text(question)
    qn = match_norm(question)
    facets = [name for name,pat in FACET_PATTERNS if re.search(pat, qn, flags=re.I)]
    if 'winner' in facets and 'outcome' not in facets:
        facets.append('outcome')
    if not facets:
        facets = ['general']
    years = sorted({int(x) for x in re.findall(r'(?<!\d)(\d{3,4})(?!\d)', question)})
    return {'question':question,'facets':facets,'years':years,'is_multi_part':len([x for x in facets if x!='time'])>=2}

def plan_query_variants_core(question: str) -> List[str]:
    analysis = analyze_question_core(question)
    base = clean_text(question)
    variants = [base]
    for facet in analysis['facets']:
        suffix = FACET_QUERY_SUFFIX.get(facet)
        if suffix:
            variants.append(clean_text(base + ' ' + suffix))
        if len(variants) >= MAX_QUERY_VARIANTS:
            break
    return list(dict.fromkeys(variants))[:MAX_QUERY_VARIANTS]

@tool
def history_question_analyzer(question: str) -> dict:
    'Phân tích câu hỏi thành các facet cần trả lời.'
    return analyze_question_core(question)

@tool
def retrieval_query_planner(question: str) -> list:
    'Tạo các query retrieval có mục tiêu từ câu hỏi gốc.'
    return plan_query_variants_core(question)

def context_covers_facet(chunk: Dict[str, Any], facet: str) -> bool:
    if facet in {'general','compare','time'}:
        return False
    md = chunk.get('metadata') or {}
    md_facets = {match_norm(x) for x in (md.get('content_facets') or []) if clean_text(x)}
    if FACET_TO_METADATA.get(facet,set()) & md_facets:
        return True
    body = match_norm(f"{chunk.get('title','')} {chunk.get('text','')}")
    return any(term in body for term in FACET_COVERAGE_TERMS.get(facet,[]))

def metadata_bonus(question: str, c: Dict[str, Any], analysis: Optional[Dict[str,Any]]=None) -> Tuple[float,List[str]]:
    analysis = analysis or analyze_question_core(question)
    md = c.get('metadata') or {}
    qn = match_norm(question)
    qyears = {int(x) for x in re.findall(r'(?<!\d)(\d{3,4})(?!\d)', question)}
    bonus, hits = 0.0, []

    years = {int(y) for y in md.get('years',[]) if str(y).isdigit()}
    year_hits = sorted(qyears & years)
    if year_hits:
        bonus += min(0.07, 0.035*len(year_hits))
        hits.append('years=' + ','.join(map(str,year_hits)))

    field_weights = {'people':0.055,'events':0.045,'documents':0.055,'dynasties':0.055,'locations':0.030,'periods':0.030,'topics':0.012}
    for field,weight in field_weights.items():
        matched=[]
        for item in md.get(field,[]) or []:
            ni=match_norm(item)
            if len(ni)>=4 and re.search(rf'(?<![a-z0-9]){re.escape(ni)}(?![a-z0-9])', qn):
                matched.append(str(item))
        if matched:
            bonus += weight*min(2,len(matched))
            hits.append(f"{field}="+'|'.join(matched[:2]))

    md_content = {match_norm(x) for x in (md.get('content_facets') or []) if clean_text(x)}
    for facet in analysis.get('facets',[]):
        if FACET_TO_METADATA.get(facet,set()) & md_content:
            bonus += INTENT_FACET_BONUS
            hits.append('intent_facet='+facet)

    return min(METADATA_MAX_BONUS,bonus), hits

print("✅ Tools ready: history_question_analyzer, retrieval_query_planner")


Loading reranker: BAAI/bge-reranker-v2-m3


config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

✅ Tools ready: history_question_analyzer, retrieval_query_planner


## 8. Hybrid retrieval v2 — multi-query + diversity

Luồng retrieval:

```text
FAISS Top-K
   +
BM25 Top-K
   ↓
RRF fusion
   ↓
Top 20 candidates
   ↓
Cross-Encoder reranker
   ↓
Metadata soft boost
   ↓
Final context
```

Điểm quan trọng:

- RRF chọn đúng **20 ứng viên**.
- Metadata không dùng để filter cứng.
- OOD guard được chạy trước khi đưa context vào Qwen.
- Retrieval result giữ score và metadata-hit để tiện debug.


In [8]:
# Cell 8 — Multi-query Hybrid Retrieval + RRF Top 20 + diversity

def dense_search(question: str, k: int=DENSE_FETCH_K) -> List[Tuple[int,float]]:
    q_emb = embedder.encode([query_for_embedding(question)], convert_to_numpy=True, normalize_embeddings=True).astype('float32')
    scores,idxs = faiss_index.search(q_emb, min(k,faiss_index.ntotal))
    return [(int(i),float(s)) for i,s in zip(idxs[0],scores[0]) if int(i)>=0]

def bm25_search(question: str, k: int=BM25_FETCH_K) -> List[Tuple[int,float]]:
    qt = bm25s.tokenize([match_norm(question)], stopwords=None, stemmer=None)
    idxs,scores = bm25_retriever.retrieve(qt, k=min(k,len(chunks)))
    return [(int(i),float(s)) for i,s in zip(np.asarray(idxs[0]).tolist(), np.asarray(scores[0]).tolist())]

def multi_query_rrf(runs: List[Dict[str,Any]], top_k: int=RRF_TOP_K) -> List[Dict[str,Any]]:
    fused=defaultdict(float)
    info=defaultdict(lambda:{'retrieval_hits':[],'best_dense_score':None,'best_bm25_score':None})
    for q_idx,run in enumerate(runs):
        w=1.0 if q_idx==0 else QUERY_EXPANSION_WEIGHT
        for rank,(idx,score) in enumerate(run['dense'],1):
            fused[idx]+=w/(RRF_K+rank)
            info[idx]['retrieval_hits'].append(f'dense:q{q_idx}@{rank}')
            cur=info[idx]['best_dense_score']
            if cur is None or score>cur: info[idx]['best_dense_score']=float(score)
        for rank,(idx,score) in enumerate(run['bm25'],1):
            fused[idx]+=w/(RRF_K+rank)
            info[idx]['retrieval_hits'].append(f'bm25:q{q_idx}@{rank}')
            cur=info[idx]['best_bm25_score']
            if cur is None or score>cur: info[idx]['best_bm25_score']=float(score)
    ordered=sorted(fused,key=fused.get,reverse=True)[:top_k]
    out=[]
    for idx in ordered:
        c=dict(chunks[idx]); c['_corpus_idx']=idx; c['rrf_score']=float(fused[idx]); c.update(info[idx]); out.append(c)
    return out

def minmax(values: List[float]) -> np.ndarray:
    arr=np.asarray(values,dtype=np.float32)
    if len(arr)==0: return arr
    lo,hi=float(arr.min()),float(arr.max())
    if hi-lo<1e-9: return np.ones_like(arr)*0.5
    return (arr-lo)/(hi-lo)

def select_diverse_contexts(candidates, analysis, final_k):
    if not candidates: return []
    if not ENABLE_CONTEXT_DIVERSITY: return candidates[:final_k]
    selected=[]; selected_ids=set(); title_counts=Counter()
    def tkey(c): return match_norm(c.get('title','')) or str(c.get('chunk_id',''))
    def can_add(c,cap=MAX_CHUNKS_PER_TITLE):
        return str(c['chunk_id']) not in selected_ids and title_counts[tkey(c)]<cap
    def add(c):
        selected.append(c); selected_ids.add(str(c['chunk_id'])); title_counts[tkey(c)]+=1

    for facet in analysis.get('facets',[]):
        if facet in {'general','compare','time'}: continue
        for c in candidates:
            if can_add(c) and context_covers_facet(c,facet):
                add(c); break
        if len(selected)>=final_k: break

    for c in candidates:
        if len(selected)>=final_k: break
        if can_add(c): add(c)

    if len(selected)<final_k:
        for c in candidates:
            if len(selected)>=final_k: break
            if str(c['chunk_id']) not in selected_ids: add(c)

    selected.sort(key=lambda x:x.get('final_retrieval_score',0.0), reverse=True)
    return selected[:final_k]

def context_title_diversity(contexts):
    if not contexts: return 0.0
    return len({match_norm(c.get('title','')) for c in contexts})/len(contexts)

def hybrid_retrieve(question: str, final_k: int=FINAL_CONTEXT_K, analysis=None, query_variants=None) -> Dict[str,Any]:
    question=clean_text(question)
    analysis=analysis or history_question_analyzer.invoke({'question':question})
    query_variants=query_variants or retrieval_query_planner.invoke({'question':question})
    intent=intent_scores(question)

    if intent['explicit_ood'] and intent['margin']<OOD_ANCHOR_MARGIN:
        return {'question':question,'is_ood':True,'ood_reason':'explicit_ood+anchor_guard',
                'intent':{k:v for k,v in intent.items() if k!='query_embedding'},
                'analysis':analysis,'query_variants':query_variants,'candidates20':[],'final_context':[],
                'max_dense':None,'context_title_diversity':0.0,
                'tool_trace':['question_analyzer','query_planner','ood_guard:block']}

    runs=[{'query':q,'dense':dense_search(q),'bm25':bm25_search(q)} for q in query_variants]
    original_dense=runs[0]['dense'] if runs else []
    max_dense=original_dense[0][1] if original_dense else -1.0

    if intent['margin']<SECONDARY_OOD_MARGIN and max_dense<SECONDARY_MIN_DENSE:
        return {'question':question,'is_ood':True,'ood_reason':'weak_history_retrieval+anchor_guard',
                'intent':{k:v for k,v in intent.items() if k!='query_embedding'},
                'analysis':analysis,'query_variants':query_variants,'candidates20':[],'final_context':[],
                'max_dense':float(max_dense),'context_title_diversity':0.0,
                'tool_trace':['question_analyzer','query_planner','multi_query_retrieval','ood_guard:block']}

    candidates=multi_query_rrf(runs, top_k=RRF_TOP_K)
    if not candidates:
        return {'question':question,'is_ood':False,'ood_reason':'',
                'intent':{k:v for k,v in intent.items() if k!='query_embedding'},
                'analysis':analysis,'query_variants':query_variants,'candidates20':[],'final_context':[],
                'max_dense':float(max_dense),'context_title_diversity':0.0,
                'tool_trace':['question_analyzer','query_planner','multi_query_retrieval:no_candidates']}

    pairs=[[question,clean_text(f"{c.get('title','')}\n{c.get('text','')}")] for c in candidates]
    rr_scores=np.asarray(reranker.predict(pairs,batch_size=RERANK_BATCH_SIZE,show_progress_bar=False,convert_to_numpy=True)).reshape(-1).astype(float)
    rr_norm=minmax(rr_scores.tolist())
    rrf_norm=minmax([c['rrf_score'] for c in candidates])

    for j,c in enumerate(candidates):
        bonus,hits=metadata_bonus(question,c,analysis=analysis)
        c['reranker_score']=float(rr_scores[j]); c['reranker_norm']=float(rr_norm[j]); c['rrf_norm']=float(rrf_norm[j])
        c['metadata_bonus']=float(bonus); c['metadata_hits']=hits
        c['final_retrieval_score']=float(0.72*rr_norm[j]+0.28*rrf_norm[j]+bonus)

    candidates.sort(key=lambda x:x['final_retrieval_score'], reverse=True)
    candidates=candidates[:RRF_TOP_K]
    final_context=select_diverse_contexts(candidates,analysis,final_k)

    return {'question':question,'is_ood':False,'ood_reason':'',
            'intent':{k:v for k,v in intent.items() if k!='query_embedding'},
            'analysis':analysis,'query_variants':query_variants,'candidates20':candidates,'final_context':final_context,
            'max_dense':float(max_dense),'context_title_diversity':context_title_diversity(final_context),
            'tool_trace':['question_analyzer','query_planner',f'multi_query_faiss_bm25:{len(query_variants)}q',
                          'weighted_rrf:top20','cross_encoder_reranker','metadata_intent_boost','evidence_diversity']}

_smoke=hybrid_retrieve('Khởi nghĩa Lam Sơn diễn ra trong bối cảnh nào và kết quả ra sao?')
print('OOD:',_smoke['is_ood'])
print('Facets:',_smoke['analysis']['facets'])
print('Query variants:',_smoke['query_variants'])
for n,c in enumerate(_smoke['final_context'],1):
    print(n,c['chunk_id'],'|',c.get('title'),'|',round(c['final_retrieval_score'],4),'|',c.get('metadata_hits'))
print('Title diversity:',round(_smoke['context_title_diversity'],3))


Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

OOD: False
Facets: ['context', 'outcome']
Query variants: ['Khởi nghĩa Lam Sơn diễn ra trong bối cảnh nào và kết quả ra sao?', 'Khởi nghĩa Lam Sơn diễn ra trong bối cảnh nào và kết quả ra sao? bối cảnh hoàn cảnh trước khi nguyên nhân dẫn tới', 'Khởi nghĩa Lam Sơn diễn ra trong bối cảnh nào và kết quả ra sao? kết quả kết thúc thắng lợi thất bại hậu quả hệ quả']
1 hf_wikipedia_khởi_nghĩa_lam_sơn_ở_nghệ_an_0000_eb9c44ad9c2a | Khởi nghĩa Lam Sơn ở Nghệ An | 0.9923 | ['events=Khởi nghĩa Lam Sơn', 'topics=khởi nghĩa', 'intent_facet=context']
2 hf_wikipedia_khởi_nghĩa_lam_sơn_0000_beba19ff15c6 | Khởi nghĩa Lam Sơn | 0.8776 | ['events=Khởi nghĩa Lam Sơn', 'locations=Lam Sơn', 'topics=khởi nghĩa']
3 hf_wikipedia_lam_sơn_thực_lục_0000_a0866c94c690 | Lam Sơn thực lục | 0.8047 | ['topics=khởi nghĩa', 'intent_facet=context']
4 hf_wikipedia_lê_thái_tổ_0004_f860bb5b2be9 | Lê Thái Tổ | 0.7111 | ['events=Khởi nghĩa Lam Sơn', 'topics=khởi nghĩa']
5 hf_wikipedia_khởi_nghĩa_lam_sơn_0001_a75516f8b5e9 | Khở

## 9. Load Qwen và merge trọng số Phase 1 → Phase 2

Cell này giữ đúng thứ tự như Phase 7:

```text
Qwen2.5-3B-Instruct
        ↓
load Phase 1 LoRA
        ↓
merge_and_unload()
        ↓
load Phase 2 LoRA
        ↓
merge_and_unload()
```

Các variant benchmark:

- `vanilla`
- `stage1`
- `stage12`

### GPU

**A100 được khuyên dùng** vì lúc benchmark notebook có thể phải load nhiều variant tuần tự trong cùng runtime.

L4 thường vẫn ổn. T4 dễ sát giới hạn VRAM hơn.


In [9]:
# Cell 9 — Qwen loaders + dynamic prompt rules + answer tools

PHASE1_ADAPTER_DIR=resolve_adapter_dir('phase1',PHASE1_ADAPTER_CANDIDATES)
PHASE2_ADAPTER_DIR=resolve_adapter_dir('phase2',PHASE2_ADAPTER_CANDIDATES)
print('PHASE1_ADAPTER_DIR:',PHASE1_ADAPTER_DIR)
print('PHASE2_ADAPTER_DIR:',PHASE2_ADAPTER_DIR)

IM_START='<|im_start|>'; IM_END='<|im_end|>'
DEFAULT_SYSTEM=('Bạn là trợ lý AI chuyên về lịch sử Việt Nam. Trả lời trực tiếp đúng trọng tâm, rõ ràng và chính xác. '
                'Không mở đầu bằng các câu rập khuôn như "Theo tài liệu". Nếu không đủ cơ sở để khẳng định, nói rõ mức độ không chắc chắn.')

def load_tokenizer():
    tok=AutoTokenizer.from_pretrained(MODEL_ID,trust_remote_code=True,use_fast=True)
    if tok.pad_token_id is None: tok.pad_token=tok.eos_token
    tok.padding_side='left'
    return tok

def load_base_model():
    dtype=torch.bfloat16 if torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0]>=8 else torch.float16
    if torch.cuda.is_available() and GPU_VRAM_GB>=22: device_map={'':0}
    elif torch.cuda.is_available(): device_map='auto'
    else: device_map=None
    return AutoModelForCausalLM.from_pretrained(MODEL_ID,dtype=dtype,device_map=device_map,trust_remote_code=True)

def load_model_variant(variant: str):
    variant=variant.lower()
    if variant not in {'vanilla','stage1','stage12'}: raise ValueError(variant)
    tok=load_tokenizer(); model=load_base_model(); model.config.pad_token_id=tok.pad_token_id
    if variant in {'stage1','stage12'}:
        print('Loading Phase1 adapter → merge...')
        model=PeftModel.from_pretrained(model,PHASE1_ADAPTER_DIR,is_trainable=False).merge_and_unload(); cleanup_cuda()
    if variant=='stage12':
        print('Loading Phase2 adapter on Phase1-merged model → merge...')
        model=PeftModel.from_pretrained(model,PHASE2_ADAPTER_DIR,is_trainable=False).merge_and_unload(); cleanup_cuda()
    model.eval(); model.config.use_cache=True; model.config.pad_token_id=tok.pad_token_id
    if TEMPERATURE<=0:
        try:
            model.generation_config.do_sample=False
            model.generation_config.temperature=None
            model.generation_config.top_p=None
            model.generation_config.top_k=None
        except Exception: pass
    return model,tok

@torch.inference_mode()
def generate_raw(model,tok,prompt: str,max_new_tokens: int=MAX_NEW_TOKENS) -> str:
    device=get_model_device(model)
    inputs=tok(prompt,return_tensors='pt',add_special_tokens=False).to(device)
    im_end_id=tok.convert_tokens_to_ids(IM_END)
    eos_ids=[tok.eos_token_id]
    if isinstance(im_end_id,int) and im_end_id>=0: eos_ids.append(im_end_id)
    kwargs=dict(**inputs,max_new_tokens=max_new_tokens,do_sample=TEMPERATURE>0,
                repetition_penalty=REPETITION_PENALTY,eos_token_id=eos_ids,pad_token_id=tok.pad_token_id)
    if TEMPERATURE>0:
        kwargs['temperature']=TEMPERATURE; kwargs['top_p']=TOP_P
    out=model.generate(**kwargs)
    gen=out[0][inputs['input_ids'].shape[-1]:]
    return tok.decode(gen,skip_special_tokens=False)

def clean_generated(text: str,tok) -> str:
    text=clean_text(text)
    for marker in [IM_END,f'{IM_START}user',f'{IM_START}system',f'{IM_START}assistant']:
        if marker and marker in text: text=text.split(marker,1)[0]
    for special in [IM_START,IM_END,tok.eos_token or '',tok.pad_token or '']:
        if special: text=text.replace(special,'')
    return clean_text(text)

def polish_answer_style_core(answer: str) -> str:
    answer=clean_text(answer)
    answer=re.sub(r'^(?:(?:theo|dựa trên|căn cứ(?: vào)?)\s+(?:các\s+)?(?:tài liệu|đoạn tư liệu)(?:\s+được\s+(?:cung cấp|truy xuất))?\s*[:,]?\s*)+','',answer,flags=re.I).strip()
    answer=re.sub(r'\bChunk\s+(?:cũng\s+)?nêu(?:\s+rằng)?\s*[:,]?\s*','Ngoài ra, ',answer,flags=re.I)
    answer=re.sub(r'\bChunk\s+này\s+(?:cho biết|mô tả|nêu)\s*[:,]?\s*','Cụ thể, ',answer,flags=re.I)
    answer=re.sub(r'\bcác?\s+chunk\b','các đoạn tư liệu',answer,flags=re.I)
    answer=re.sub(r'\bchunk\b','đoạn tư liệu',answer,flags=re.I)
    answer=clean_text(answer)
    if answer: answer=answer[0].upper()+answer[1:]
    return answer

@tool
def answer_style_polisher(answer: str) -> str:
    'Bỏ boilerplate và từ kỹ thuật chunk khỏi câu trả lời user-facing.'
    return polish_answer_style_core(answer)

def build_dynamic_answer_rules(question: str) -> List[str]:
    analysis=analyze_question_core(question); facets=analysis['facets']
    rules=[
        'Trả lời trực tiếp trọng tâm ngay từ câu đầu; không chỉ tóm tắt tài liệu.',
        'Chỉ dùng thông tin được các đoạn tài liệu bên dưới hỗ trợ; không tự bổ sung kiến thức ngoài evidence.',
        'Không mở đầu bằng "Theo tài liệu", "Dựa trên tài liệu" hoặc cách nói tương tự.',
        'Không dùng từ "chunk" trong câu trả lời.',
        'Không lặp lại nguyên câu hỏi.',
        'Không suy luận quan hệ nhân vật, triều đại, phe phái hoặc niên đại nếu evidence không nêu đủ rõ.',
        'Nếu evidence chưa đủ để kết luận chắc chắn, nói rõ phần nào chưa đủ thay vì đoán.',
        'Nguồn được dùng chỉ được chứa chunk_id thực sự có trong Tài liệu tham khảo.',
    ]
    if analysis['is_multi_part']:
        rules.append('Câu hỏi có nhiều ý: phải trả lời đủ từng ý; có thể dùng nhãn ngắn Bối cảnh/Kết quả/Ý nghĩa nếu giúp rõ hơn.')
    if 'winner' in facets:
        rules += [
            'Câu hỏi hỏi bên thắng: phải trả lời trực tiếp thắng/thua/không thể kết luận ở câu đầu.',
            'Không được suy luận "bên phát động = bên chiến thắng".',
            'Nếu evidence cho thấy kết quả khác nhau theo quân sự, chiến thuật, chiến lược hoặc chính trị, phải tách các khía cạnh đó.',
        ]
    if 'compare' in facets:
        rules += [
            'Câu hỏi so sánh: nêu riêng vai trò/đặc điểm của từng bên, sau đó mới chỉ ra điểm giống/khác hoặc kết luận.',
            'Không gán nhân vật, sự kiện hay đặc điểm của đối tượng thứ nhất sang đối tượng thứ hai.',
        ]
    if 'context' in facets: rules.append('Phải nêu hoàn cảnh/bối cảnh dẫn tới sự kiện, không chỉ mô tả sự kiện.')
    if 'outcome' in facets: rules.append('Phải nêu kết quả/kết cục hoặc hệ quả trực tiếp nếu câu hỏi yêu cầu.')
    if 'significance' in facets: rules.append('Phải giải thích ý nghĩa/tác động, không chỉ kể diễn biến.')
    if 'process' in facets: rules.append('Nếu hỏi diễn biến, trình bày các mốc/chặng chính theo trật tự thời gian khi evidence hỗ trợ.')
    if 'content' in facets: rules.append('Nếu hỏi nội dung văn kiện/hiệp định, ưu tiên các điểm chính và tách chúng khỏi phần hệ quả.')
    return rules

def build_plain_prompt(question: str) -> str:
    return f'{IM_START}system\n{DEFAULT_SYSTEM}{IM_END}\n{IM_START}user\n{clean_text(question)}{IM_END}\n{IM_START}assistant\n'

def extract_plain_answer(raw: str,tok) -> str:
    raw=clean_generated(raw,tok)
    if '<final>' in raw:
        part=raw.split('<final>',1)[1]
        if '</final>' in part: part=part.split('</final>',1)[0]
        return polish_answer_style_core(part)
    return polish_answer_style_core(raw)

def build_context_text(contexts,chars_per_chunk):
    return '\n\n'.join([f"[{c['chunk_id']}] {clean_text(c.get('title',''))}\n{short_text(c.get('text',''),chars_per_chunk)}" for c in contexts])

def build_rag_user_text(question,contexts,chars_per_chunk):
    rules=build_dynamic_answer_rules(question)
    rule_text='\n'.join(f'{j}. {r}' for j,r in enumerate(rules,1))
    return (f'Câu hỏi:\n{clean_text(question)}\n\nYêu cầu bắt buộc:\n{rule_text}\n\n'
            f'Định dạng đầu ra bắt buộc:\nNguồn được dùng: [chunk_id_1, chunk_id_2]\nTrả lời: <câu trả lời trực tiếp>\n\n'
            f'Tài liệu tham khảo:\n{build_context_text(contexts,chars_per_chunk)}').strip()

def build_rag_prompt(user_text: str) -> str:
    return f'{IM_START}user\n{user_text}{IM_END}\n{IM_START}assistant\n'

def fit_rag_prompt(tok,question,contexts):
    used=list(contexts); chars=MAX_CHARS_PER_CHUNK
    while used:
        prompt=build_rag_prompt(build_rag_user_text(question,used,chars))
        n=len(tok(prompt,add_special_tokens=False)['input_ids'])
        if n<=MAX_INPUT_TOKENS:
            return prompt,used,{'input_tokens':n,'chars_per_chunk':chars,'n_context':len(used)}
        if chars>MIN_CHARS_PER_CHUNK: chars=max(MIN_CHARS_PER_CHUNK,int(chars*0.75))
        else: used=used[:-1]
    prompt=build_rag_prompt(build_rag_user_text(question,[],0))
    return prompt,[],{'input_tokens':len(tok(prompt,add_special_tokens=False)['input_ids']),'chars_per_chunk':0,'n_context':0}

SOURCE_BLOCK_RE=re.compile(r'Nguồn được dùng\s*:\s*(.*?)(?:\n\s*\n|\n\s*Trả lời\s*:|$)',re.I|re.S)
ANSWER_SPLIT_RE=re.compile(r'Trả lời\s*:',re.I)

def parse_rag_output(raw,tok):
    cleaned=clean_generated(raw,tok); source_ids=[]
    m=SOURCE_BLOCK_RE.search(cleaned)
    if m:
        block=m.group(1); groups=re.findall(r'\[([^\[\]]*)\]',block)
        if not groups and block.strip(): groups=[block]
        for g in groups:
            for x in re.split(r'[,;\n]+',g):
                x=x.strip().strip('[]"\' ')
                if x: source_ids.append(x)
    source_ids=list(dict.fromkeys(source_ids))
    parts=ANSWER_SPLIT_RE.split(cleaned,maxsplit=1)
    answer=clean_text(parts[1] if len(parts)>1 else cleaned)
    answer=answer_style_polisher.invoke({'answer':answer})
    return {'raw_output':cleaned,'source_ids':source_ids,'answer':answer,
            'format_ok':bool(re.search(r'Nguồn được dùng\s*:',cleaned,re.I) and re.search(r'Trả lời\s*:',cleaned,re.I))}

REFUSAL_PATTERNS=[
    r'không đủ (?:thông tin|bằng chứng|dữ liệu)',
    r'tài liệu (?:được cung cấp )?(?:không|chưa) (?:nêu|cho biết|cung cấp)',
    r'không thể trả lời (?:chắc chắn|chính xác)?',
    r'ngoài phạm vi',
    r'không liên quan đến lịch sử việt nam',
]
def is_refusal(text): return any(re.search(p,clean_text(text),flags=re.I) for p in REFUSAL_PATTERNS)

def critique_answer_core(question,answer):
    analysis=analyze_question_core(question); facets=analysis['facets']; a=match_norm(answer); issues=[]
    if not clean_text(answer): return {'pass':False,'issues':['empty_answer'],'facets':facets}
    if is_refusal(answer): return {'pass':True,'issues':[],'facets':facets}
    if re.match(r'\s*(theo|dua tren|can cu).*tai lieu',a): issues.append('boilerplate_opening')
    if re.search(r'\bchunk\b',a): issues.append('technical_chunk_word')
    if 'winner' in facets and not any(x in a for x in ['gianh chien thang','thang loi','that bai','gianh uu the','uu the quan su','khong the ket luan','khong co mot ben','khong the coi','khong co ben nao']):
        issues.append('winner_not_answered_directly')
    if 'outcome' in facets and not any(x in a for x in ['ket qua','ket thuc','thang loi','that bai','cham dut','gianh','buoc','thanh lap','thoai vi','tao co so','dan den','he qua']):
        issues.append('missing_outcome')
    if 'significance' in facets and not any(x in a for x in ['y nghia','danh dau','khang dinh','mo ra','gop phan','tac dong','vai tro','tao tien de','cung co']):
        issues.append('missing_significance')
    if 'context' in facets and not any(x in a for x in ['boi canh','hoan canh','truoc khi','sau khi','trong khi','xam luoc','khong chien','duoi su','do ','vi ','khi ']):
        issues.append('missing_context')
    if 'compare' in facets and not any(x in a for x in ['trong khi','con ','deu ','khac','giong','so voi','ve mat','tuong dong']):
        issues.append('comparison_not_explicit')
    if 'content' in facets and not any(x in a for x in ['noi dung','quy dinh','dieu khoan','ngung ban','cam ket','trao tra','tong tuyen cu']):
        issues.append('missing_document_content')
    if analysis['is_multi_part'] and len(re.findall(r"[0-9A-Za-zÀ-ỹĐđ]+",clean_text(answer)))<35:
        issues.append('multi_part_answer_too_short')
    issues=list(dict.fromkeys(issues))
    return {'pass':not issues,'issues':issues,'facets':facets}

@tool
def answer_quality_critic(question: str, answer: str) -> dict:
    'Kiểm tra answer có trực tiếp và đủ các facet được hỏi hay không.'
    return critique_answer_core(question,answer)

ISSUE_INSTRUCTIONS={
    'empty_answer':'Câu trả lời bị rỗng.',
    'boilerplate_opening':'Bỏ cách mở đầu rập khuôn như "Theo tài liệu".',
    'technical_chunk_word':'Không dùng từ kỹ thuật "chunk".',
    'winner_not_answered_directly':'Phải trả lời trực tiếp bên nào thắng/giành ưu thế, hoặc nói evidence không cho phép kết luận tuyệt đối.',
    'missing_outcome':'Bổ sung phần kết quả/kết cục được evidence hỗ trợ.',
    'missing_significance':'Bổ sung phần ý nghĩa/tác động được evidence hỗ trợ.',
    'missing_context':'Bổ sung bối cảnh/hoàn cảnh dẫn tới sự kiện.',
    'comparison_not_explicit':'So sánh rõ từng bên và điểm giống/khác; không trộn thông tin.',
    'missing_document_content':'Nêu các nội dung/điều khoản chính được evidence hỗ trợ.',
    'multi_part_answer_too_short':'Câu hỏi có nhiều ý; trả lời đủ từng ý.',
    'invalid_source_id':'Chỉ dùng chunk_id có trong evidence.',
    'missing_source':'Phải ghi ít nhất một source nếu đưa ra factual answer.',
    'unsupported_year':'Loại hoặc sửa niên đại không có trong evidence.',
}

def build_rewrite_user_text(question,contexts,chars,draft,issues):
    issue_text='\n'.join('- '+ISSUE_INSTRUCTIONS.get(x,x) for x in issues)
    rules=build_dynamic_answer_rules(question)
    rule_text='\n'.join(f'{j}. {r}' for j,r in enumerate(rules,1))
    return (f'Câu hỏi:\n{clean_text(question)}\n\nCâu trả lời nháp cần sửa:\n{clean_text(draft)}\n\n'
            f'Các lỗi cần sửa:\n{issue_text}\n\nYêu cầu bắt buộc:\n{rule_text}\n'
            f'{len(rules)+1}. Chỉ sửa bằng evidence bên dưới; không giữ claim trong bản nháp nếu evidence không hỗ trợ.\n\n'
            f'Định dạng đầu ra bắt buộc:\nNguồn được dùng: [chunk_id_1, chunk_id_2]\nTrả lời: <bản trả lời đã sửa>\n\n'
            f'Tài liệu tham khảo:\n{build_context_text(contexts,chars)}').strip()

def fit_rewrite_prompt(tok,question,contexts,draft,issues):
    used=list(contexts); chars=min(MAX_CHARS_PER_CHUNK,1500)
    while used:
        prompt=build_rag_prompt(build_rewrite_user_text(question,used,chars,draft,issues))
        n=len(tok(prompt,add_special_tokens=False)['input_ids'])
        if n<=MAX_INPUT_TOKENS: return prompt,used,{'input_tokens':n,'chars_per_chunk':chars,'n_context':len(used)}
        if chars>MIN_CHARS_PER_CHUNK: chars=max(MIN_CHARS_PER_CHUNK,int(chars*0.75))
        else: used=used[:-1]
    prompt=build_rag_prompt(build_rewrite_user_text(question,[],0,draft,issues))
    return prompt,[],{'input_tokens':len(tok(prompt,add_special_tokens=False)['input_ids']),'chars_per_chunk':0,'n_context':0}

print("✅ Prompt + style/critic tools ready.")


PHASE1_ADAPTER_DIR: /content/drive/MyDrive/vn_history_model_backups/qwen_vnhistory_phase1_best_adapter
PHASE2_ADAPTER_DIR: /content/drive/MyDrive/vn_history_model_backups/qwen_vnhistory_phase6_rag_best_adapter
✅ Prompt + style/critic tools ready.


## 10. LangChain tool orchestration + grounded repair

Load model đã merge đủ hai stage và dựng pipeline:

```text
normalize question
→ hybrid retrieval
→ guarded generation
→ source validation
→ unsupported-year validation
→ final answer
```

Các guard chính:

- chặn câu hỏi ngoài phạm vi;
- không cho model cite `chunk_id` ngoài context;
- yêu cầu source khi trả lời;
- chặn năm không xuất hiện trong evidence;
- trả lời “không đủ bằng chứng” khi cần.

Đây là **cấu hình đầy đủ nhất** dùng cho inference thực tế.


In [10]:
# Cell 10 — Final model + deterministic LangChain tool orchestration + evidence-only repair

print('Loading FINAL Stage1+Stage2 merged model...')
generation_model,generation_tokenizer=load_model_variant('stage12')
print('Final model device:',get_model_device(generation_model))

SAFE_OOD_ANSWER='Câu hỏi này nằm ngoài phạm vi hệ thống lịch sử Việt Nam, nên tôi không trả lời bằng corpus hiện tại.'
SAFE_INSUFFICIENT_ANSWER='Không đủ bằng chứng trong các tài liệu truy xuất để trả lời chắc chắn câu hỏi này.'

def extract_year_set(text):
    return {int(x) for x in re.findall(r'(?<!\d)(\d{3,4})(?!\d)',clean_text(text))}

def answer_support_score(answer,source_chunks):
    if not answer or not source_chunks: return None
    a=embedder.encode(['query: '+answer],convert_to_numpy=True,normalize_embeddings=True).astype('float32')[0]
    p=embedder.encode([passage_for_embedding(c) for c in source_chunks],convert_to_numpy=True,normalize_embeddings=True).astype('float32')
    return float(np.max(p@a))

def validate_parsed_answer(parsed,used_context):
    allowed={str(c['chunk_id']) for c in used_context}
    model_ids=[str(x) for x in parsed.get('source_ids',[])]
    valid=[x for x in model_ids if x in allowed]; invalid=[x for x in model_ids if x not in allowed]
    evidence=[chunk_by_id[x] for x in valid if x in chunk_by_id]
    evidence_text='\n'.join(clean_text(c.get('title',''))+'\n'+clean_text(c.get('text','')) for c in evidence)
    answer=parsed.get('answer','')
    unsupported=sorted(extract_year_set(answer)-extract_year_set(evidence_text)) if evidence else sorted(extract_year_set(answer))
    guard=[]
    if invalid: guard.append('invalid_source_id')
    if STRICT_SOURCE_REQUIRED and not valid and not is_refusal(answer): guard.append('missing_source')
    if STRICT_UNSUPPORTED_YEAR_GUARD and unsupported and not is_refusal(answer): guard.append('unsupported_year')
    return {'answer':answer,'valid_ids':valid,'invalid_ids':invalid,'unsupported_years':unsupported,
            'evidence_chunks':evidence,'guard_issues':guard,'format_ok':parsed.get('format_ok',False)}

def run_generation_pass(question,contexts):
    prompt,used,budget=fit_rag_prompt(generation_tokenizer,question,contexts)
    raw=generate_raw(generation_model,generation_tokenizer,prompt,max_new_tokens=MAX_NEW_TOKENS)
    parsed=parse_rag_output(raw,generation_tokenizer)
    valid=validate_parsed_answer(parsed,used)
    crit=answer_quality_critic.invoke({'question':question,'answer':valid['answer']})
    return {'prompt':prompt,'budget':budget,'used_context':used,'raw':raw,'parsed':parsed,'validated':valid,'critique':crit}

def run_repair_pass(question,contexts,draft,reasons):
    prompt,used,budget=fit_rewrite_prompt(generation_tokenizer,question,contexts,draft,reasons)
    raw=generate_raw(generation_model,generation_tokenizer,prompt,max_new_tokens=MAX_NEW_TOKENS)
    parsed=parse_rag_output(raw,generation_tokenizer)
    valid=validate_parsed_answer(parsed,used)
    crit=answer_quality_critic.invoke({'question':question,'answer':valid['answer']})
    return {'prompt':prompt,'budget':budget,'used_context':used,'raw':raw,'parsed':parsed,'validated':valid,'critique':crit}

def full_rag_answer_from_state(state):
    question=state['question']; retrieval=state['retrieval']; analysis=state.get('analysis',retrieval.get('analysis',{}))
    started=time.perf_counter()

    if retrieval.get('is_ood'):
        return {'question':question,'answer':SAFE_OOD_ANSWER,'status':'blocked_off_topic','source_ids':[],
                'invalid_source_ids':[],'unsupported_years':[],'format_ok':True,'retrieval':retrieval,'analysis':analysis,
                'support_score':None,'quality_warnings':[],'rewrite_used':False,'repair_attempted':False,
                'tool_trace':retrieval.get('tool_trace',[])+['final_ood_guard'],'latency_sec':time.perf_counter()-started}

    contexts=retrieval.get('final_context',[])
    if not contexts:
        return {'question':question,'answer':SAFE_INSUFFICIENT_ANSWER,'status':'blocked_no_context','source_ids':[],
                'invalid_source_ids':[],'unsupported_years':[],'format_ok':True,'retrieval':retrieval,'analysis':analysis,
                'support_score':None,'quality_warnings':[],'rewrite_used':False,'repair_attempted':False,
                'tool_trace':retrieval.get('tool_trace',[])+['no_context_guard'],'latency_sec':time.perf_counter()-started}

    first=run_generation_pass(question,contexts)
    first_valid=first['validated']; first_crit=first['critique']
    reasons=list(dict.fromkeys(first_valid['guard_issues']+first_crit.get('issues',[])))
    chosen=first; rewrite_used=False; repair_attempted=False

    if ENABLE_COMPLETENESS_REWRITE and reasons and MAX_REWRITE_ATTEMPTS>0 and not is_refusal(first_valid['answer']):
        repair_attempted=True
        repaired=run_repair_pass(question,contexts,first_valid['answer'],reasons)
        rv=repaired['validated']; rc=repaired['critique']
        structurally_better=not rv['guard_issues']
        quality_better=len(rc.get('issues',[]))<len(first_crit.get('issues',[]))
        if structurally_better and (quality_better or bool(first_valid['guard_issues'])):
            chosen=repaired; rewrite_used=True

    valid=chosen['validated']; crit=chosen['critique']; answer=valid['answer']; status='ok'
    if valid['guard_issues']:
        if 'invalid_source_id' in valid['guard_issues']: status='blocked_invalid_source_id'
        elif 'missing_source' in valid['guard_issues']: status='blocked_missing_source'
        elif 'unsupported_year' in valid['guard_issues']: status='blocked_unsupported_year'
        else: status='blocked_guard'
        answer=SAFE_INSUFFICIENT_ANSWER; source_ids=[]
    else:
        source_ids=valid['valid_ids']
        if crit.get('issues'): status='ok_with_quality_warning'

    evidence=[chunk_by_id[x] for x in source_ids if x in chunk_by_id]
    support=answer_support_score(answer,evidence) if evidence else None
    trace=retrieval.get('tool_trace',[])+['qwen_generation','style_polisher','source_year_guard','answer_quality_critic']
    if repair_attempted: trace.append('evidence_only_repair')
    if rewrite_used: trace.append('repair_accepted')
    elif repair_attempted: trace.append('repair_rejected_keep_first')

    return {'question':question,'answer':answer,'status':status,'source_ids':source_ids,
            'model_source_ids':chosen['parsed'].get('source_ids',[]),'invalid_source_ids':valid['invalid_ids'],
            'unsupported_years':valid['unsupported_years'],'format_ok':chosen['parsed'].get('format_ok',False),
            'raw_output':chosen['parsed'].get('raw_output',''),'retrieval':retrieval,'analysis':analysis,
            'prompt_budget':chosen['budget'],'support_score':support,'quality_warnings':crit.get('issues',[]),
            'rewrite_used':rewrite_used,'repair_attempted':repair_attempted,
            'initial_quality_issues':first_crit.get('issues',[]),'tool_trace':trace,
            'latency_sec':time.perf_counter()-started}

def state_analyze(s):
    return {**s,'analysis':history_question_analyzer.invoke({'question':s['question']})}

def state_plan(s):
    return {**s,'query_variants':retrieval_query_planner.invoke({'question':s['question']})}

def state_retrieve(s):
    return {**s,'retrieval':hybrid_retrieve(s['question'],analysis=s['analysis'],query_variants=s['query_variants'])}

rag_chain=(RunnableLambda(lambda q:{'question':clean_text(q)})
           | RunnableLambda(state_analyze)
           | RunnableLambda(state_plan)
           | RunnableLambda(state_retrieve)
           | RunnableLambda(full_rag_answer_from_state))

def ask_history(question: str, verbose: bool=True):
    result=rag_chain.invoke(question)
    if verbose:
        print('\nQUESTION:',result['question'])
        print('STATUS:',result['status'])
        print('ANSWER:',result['answer'])
        print('SOURCES:',result.get('source_ids',[]))
        if result.get('rewrite_used'): print('REWRITE: ✅ evidence-only repair accepted')
        elif result.get('repair_attempted'): print('REWRITE: attempted but first answer kept')
        if result.get('quality_warnings'): print('QUALITY WARNINGS:',result['quality_warnings'])
        if result.get('unsupported_years'): print('UNSUPPORTED YEARS:',result['unsupported_years'])
        if result.get('invalid_source_ids'): print('INVALID SOURCE IDS:',result['invalid_source_ids'])
        retr=result.get('retrieval',{})
        if SHOW_TOOL_TRACE:
            print('\nFacets:',retr.get('analysis',{}).get('facets',[]))
            print('Query variants:',retr.get('query_variants',[]))
            print('Tool trace:',result.get('tool_trace',[]))
            print('Context title diversity:',round(retr.get('context_title_diversity',0.0),3))
        if retr.get('final_context'):
            print('\nTop contexts:')
            for j,c in enumerate(retr['final_context'],1):
                print(f" {j}. {c['chunk_id']} | {c.get('title','')} | final={c.get('final_retrieval_score',0):.4f} | md={c.get('metadata_hits',[])}")
    return result

print('✅ LangChain deterministic tool-use RAG ready.')


Loading FINAL Stage1+Stage2 merged model...


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model-00002-of-00002.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/3.97G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Loading Phase1 adapter → merge...
Loading Phase2 adapter on Phase1-merged model → merge...
Final model device: cuda:0
✅ LangChain deterministic tool-use RAG ready.


## 11. Regression smoke tests

Test hai trường hợp:

1. một câu hỏi lịch sử hợp lệ;
2. một câu hỏi lạc đề.

Mục tiêu là kiểm tra nhanh:

- retrieval có chạy;
- model trả lời được;
- OOD guard hoạt động;
- source IDs được kiểm soát.

Nên chạy cell này trước khi chạy cả bộ manual/benchmark.


### Regression cần quan sát

Đặc biệt với Tết Mậu Thân:

- câu đầu phải trả lời trực tiếp vấn đề thắng/thua hoặc nói rõ không thể quy về một phe thắng tuyệt đối;
- không suy luận “bên phát động = bên thắng”;
- nếu evidence thể hiện nhiều lớp kết quả, nên tách quân sự / chiến lược / chính trị;
- không mở đầu bằng “Theo tài liệu”;
- không dùng từ “chunk”;
- nếu câu đầu chưa đủ ý, có thể thấy `REWRITE: ✅ evidence-only repair accepted`.


In [11]:
# Cell 11 — Regression smoke tests

print('\n### TEST A — normal history')
_ = ask_history('Chiến thắng Bạch Đằng năm 938 có ý nghĩa lịch sử gì?', verbose=True)

print('\n'+'='*110)
print('\n### TEST B — winner/directness regression')
_ = ask_history('Chiến dịch Tết Mậu Thân năm 1968 là chiến thắng của phe nào?', verbose=True)

print('\n'+'='*110)
print('\n### TEST C — off-topic')
_ = ask_history('Viết cho tôi một hàm Python sắp xếp danh sách số nguyên.', verbose=True)



### TEST A — normal history


Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

...

Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]


QUESTION: Chiến dịch Tết Mậu Thân năm 1968 là chiến thắng của phe nào?
STATUS: ok_with_quality_warning
ANSWER: Tết Mậu Thân năm 1968 là cuộc tổng tiến công và nổi dậy giành chính quyền của Quân Giải phóng miền Nam Việt Nam. Chiến dịch gây chấn động miền Nam và được mô tả là một chiến thắng có tầm vóc lớn về chiến lược.
SOURCES: ['hf_wikipedia_sự_kiện_tết_mậu_thân_0000_c92453ef84bb']
REWRITE: attempted but first answer kept
QUALITY WARNINGS: ['winner_not_answered_directly']

Facets: ['winner', 'outcome']
Query variants: ['Chiến dịch Tết Mậu Thân năm 1968 là chiến thắng của phe nào?', 'Chiến dịch Tết Mậu Thân năm 1968 là chiến thắng của phe nào? kết quả quân sự chiến thuật chiến lược chính trị bên thắng bên thua tổn thất mục tiêu', 'Chiến dịch Tết Mậu Thân năm 1968 là chiến thắng của phe nào? kết quả kết thúc thắng lợi thất bại hậu quả hệ quả']
Tool trace: ['question_analyzer', 'query_planner', 'multi_query_faiss_bm25:3q', 'weighted_rrf:top20', 'cross_encoder_reranker', 'metadata_intent

## 12. Manual test — 10 câu + quality diagnostics

Chạy 10 câu đại diện nhiều giai đoạn:

- Bạch Đằng 938;
- Lý Công Uẩn;
- Bình Ngô đại cáo;
- Lam Sơn;
- Lý / Trần;
- Cần Vương;
- Xô Viết Nghệ Tĩnh;
- Cách mạng tháng Tám;
- Genève 1954;
- ba lần kháng chiến Mông–Nguyên.

Kết quả cuối cell được gom thành DataFrame để xem:

- status;
- answer;
- sources;
- support score;
- latency.


In [12]:
# Cell 12 — Manual test 10 câu + quality diagnostics

test_questions = [
    'Chiến thắng Bạch Đằng năm 938 có ý nghĩa lịch sử gì?',
    'Việc Lý Công Uẩn dời đô ra Thăng Long năm 1010 có ý nghĩa gì?',
    'Bình Ngô đại cáo ra đời trong bối cảnh nào?',
    'Khởi nghĩa Lam Sơn diễn ra trong bối cảnh nào và kết quả ra sao?',
    'So sánh vai trò của nhà Lý và nhà Trần trong xây dựng và bảo vệ Đại Việt.',
    'Phong trào Cần Vương bùng nổ trong hoàn cảnh nào?',
    'Xô Viết Nghệ Tĩnh 1930-1931 có đặc điểm gì nổi bật?',
    'Cách mạng tháng Tám năm 1945 diễn ra như thế nào và kết quả ra sao?',
    'Hiệp định Genève năm 1954 có nội dung/kết quả gì đối với Việt Nam?',
    'Ba lần kháng chiến chống Mông - Nguyên dưới nhà Trần có đặc điểm và ý nghĩa gì?',
]

def style_spam_flag(text):
    n=match_norm(text)
    return bool(re.match(r'\s*(theo|dua tren|can cu).*tai lieu',n) or re.search(r'\bchunk\b',n))

manual_results=[]
for j,q in enumerate(test_questions,1):
    print('\n'+'='*120); print(f'MANUAL {j}/10')
    manual_results.append(ask_history(q,verbose=True))

manual_df=pd.DataFrame([{
    'question':r['question'],'status':r['status'],'answer':r['answer'],
    'sources':','.join(r.get('source_ids',[])),'support_score':r.get('support_score'),
    'rewrite_used':r.get('rewrite_used',False),'quality_warnings':'|'.join(r.get('quality_warnings',[])),
    'style_spam':style_spam_flag(r['answer']),
    'title_diversity':r.get('retrieval',{}).get('context_title_diversity'),
    'query_variant_count':len(r.get('retrieval',{}).get('query_variants',[])),
    'latency_sec':r.get('latency_sec'),
} for r in manual_results])

display(manual_df)
print('\nMANUAL SUMMARY')
print('Style spam:',int(manual_df['style_spam'].sum()),'/10')
print('Rewrite used:',int(manual_df['rewrite_used'].sum()),'/10')
print('Quality warnings remaining:',int(manual_df['quality_warnings'].astype(bool).sum()),'/10')
print('Mean support:',round(float(manual_df['support_score'].dropna().mean()),4))
print('Mean latency:',round(float(manual_df['latency_sec'].dropna().mean()),2),'sec')



MANUAL 1/10


Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

...

Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]


QUESTION: Ba lần kháng chiến chống Mông - Nguyên dưới nhà Trần có đặc điểm và ý nghĩa gì?
STATUS: ok_with_quality_warning
ANSWER: Ba cuộc kháng chiến này được xem là một trong những trang sử hào hùng nhất của dân tộc Việt Nam và là chiến công tiêu biểu của triều Trần. Hai đợt chiến sự trước và sau các đợt chiến sự là thời gian ngoại giao; chiến tranh chính thức chỉ gần 9 tháng, chia làm ba đợt.
SOURCES: ['hf_wikipedia_chiến_tranh_nguyên_mông_đại_việt_0000_b22375705ab3']
REWRITE: attempted but first answer kept
QUALITY WARNINGS: ['missing_significance']

Facets: ['significance', 'features']
Query variants: ['Ba lần kháng chiến chống Mông - Nguyên dưới nhà Trần có đặc điểm và ý nghĩa gì?', 'Ba lần kháng chiến chống Mông - Nguyên dưới nhà Trần có đặc điểm và ý nghĩa gì? ý nghĩa tác động vai trò đánh dấu mở ra góp phần', 'Ba lần kháng chiến chống Mông - Nguyên dưới nhà Trần có đặc điểm và ý nghĩa gì? đặc điểm nổi bật tính chất lực lượng hình thức']
Tool trace: ['question_analyzer', 'query

,question,status,answer,sources,support_score,rewrite_used,quality_warnings,style_spam,title_diversity,query_variant_count,latency_sec
0,Chiến thắng Bạch Đằng năm 938 có ý nghĩa lịch ...,ok_with_quality_warning,Trận Bạch Đằng năm 938 là mưu sự thành công củ...,hf_wikipedia_trận_bạch_đằng_938_0002_d5f8e1eedf68,0.882258,False,missing_significance,False,0.833333,2,10.707010
1,Việc Lý Công Uẩn dời đô ra Thăng Long năm 1010...,ok_with_quality_warning,"Gần 1 năm sau khi lên ngôi, tháng 7 âm lịch nă...",hf_wikipedia_nhà_lý_0002_830b4c6c4cba,0.870080,False,missing_significance,False,1.000000,2,8.329030
2,Bình Ngô đại cáo ra đời trong bối cảnh nào?,ok,Bình Ngô đại cáo là bài cáo viết bằng văn ngôn...,hf_wikipedia_bình_ngô_đại_cáo_0001_ffb19efce2d9,0.870410,False,,False,0.833333,2,4.146620
3,Khởi nghĩa Lam Sơn diễn ra trong bối cảnh nào ...,ok_with_quality_warning,Khởi nghĩa Lam Sơn ở Nghệ An là mảnh đất Nghệ ...,hf_wikipedia_khởi_nghĩa_lam_sơn_ở_nghệ_an_0000...,0.920772,False,missing_outcome,False,0.833333,3,13.824768
4,So sánh vai trò của nhà Lý và nhà Trần trong x...,ok_with_quality_warning,Chiến tranh Mông - Đại Việt hay Kháng chiến ch...,hf_wikipedia_chiến_tranh_nguyên_mông_đại_việt_...,0.899086,False,comparison_not_explicit,False,0.833333,3,17.431613
5,Phong trào Cần Vương bùng nổ trong hoàn cảnh nào?,ok,Phong trào Cần Vương nổ ra cuối thế kỷ 19 do T...,hf_wikipedia_pháp_thuộc_0002_ae9e59cc6436,0.863861,False,,False,0.833333,2,4.839337
6,Xô Viết Nghệ Tĩnh 1930-1931 có đặc điểm gì nổi...,ok,Xô Viết Nghệ Tĩnh là phong trào đấu tranh của ...,hf_wikipedia_xô_viết_nghệ_tĩnh_0000_d0944ade5a55,0.909477,False,,False,0.833333,2,5.903872
7,Cách mạng tháng Tám năm 1945 diễn ra như thế n...,ok,Cách mạng tháng Tám là tổng khởi nghĩa do Đảng...,hf_wikipedia_cách_mạng_tháng_tám_0002_fba40081...,0.884971,False,,False,0.833333,3,5.201536
8,Hiệp định Genève năm 1954 có nội dung/kết quả ...,ok_with_quality_warning,Hiệp định Genève 1954 xác lập Việt Nam là một ...,hf_wikipedia_hiệp_định_genève_1954_0037_60b777...,0.888661,False,missing_outcome,False,0.666667,3,13.255649
9,Ba lần kháng chiến chống Mông - Nguyên dưới nh...,ok_with_quality_warning,Ba cuộc kháng chiến này được xem là một trong ...,hf_wikipedia_chiến_tranh_nguyên_mông_đại_việt_...,0.832094,False,missing_significance,False,0.666667,3,12.379201



MANUAL SUMMARY
Style spam: 0 /10
Rewrite used: 0 /10
Quality warnings remaining: 6 /10
Mean support: 0.8822
Mean latency: 9.6 sec


## 13. Interactive QA

Cell dùng `input()` để nhập một câu hỏi bất kỳ và chạy qua **toàn bộ hệ thống RAG cuối**.

Dùng cell này khi muốn demo hoặc kiểm tra từng câu thủ công mà không cần sửa code.


In [13]:
# Cell 13 — Interactive QA

question=input('Nhập câu hỏi lịch sử Việt Nam: ').strip()
if question:
    interactive_result=ask_history(question,verbose=True)
else:
    print('Bạn chưa nhập câu hỏi.')


Nhập câu hỏi lịch sử Việt Nam: Phạm Ngọc Thảo là ai?


Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]


QUESTION: Phạm Ngọc Thảo là ai?
STATUS: ok
ANSWER: Phạm Ngọc Thảo là một cán bộ tình báo của Quân đội nhân dân Việt Nam, hoạt động dưới vỏ bọc là sĩ quan cao cấp trong Quân lực Việt Nam Cộng hòa và một chính khách có ảnh hưởng lớn ở Sài Gòn. Ông được giao nhiệm vụ thâm nhập hàng ngũ cao cấp chính quyền Sài Gòn để phục vụ mục tiêu thống nhất đất nước.
SOURCES: ['hf_wikipedia_phạm_ngọc_thảo_0003_042cd661c405']

Facets: ['general']
Query variants: ['Phạm Ngọc Thảo là ai?']
Tool trace: ['question_analyzer', 'query_planner', 'multi_query_faiss_bm25:1q', 'weighted_rrf:top20', 'cross_encoder_reranker', 'metadata_intent_boost', 'evidence_diversity', 'qwen_generation', 'style_polisher', 'source_year_guard', 'answer_quality_critic']
Context title diversity: 0.833

Top contexts:
 1. hf_wikipedia_phạm_ngọc_thảo_0003_042cd661c405 | Phạm Ngọc Thảo | final=1.0353 | md=['people=Phạm Ngọc Thảo']
 2. hf_wikipedia_phạm_ngọc_thảo_0000_f57b81132ab4 | Phạm Ngọc Thảo | final=1.0000 | md=[]
 3. hf_wikipedia_

## 14. Tạo benchmark 100 câu

Tái tạo held-out split từ Phase 2 theo seed cũ:

- **90 câu lịch sử** từ eval/test held-out;
- **10 câu off-topic** để đo khả năng từ chối.

Benchmark được lưu xuống Drive để có thể tái sử dụng.

> Các gold source IDs được kiểm tra lại với corpus Phase 8 hiện tại để tránh đánh giá sai khi chunk cũ không còn tồn tại.


In [14]:
# Cell 14 — Build benchmark 100: reconstruct Phase2 held-out 10% → random 90 + 10 OOD
from pathlib import Path

MESSAGES_PATH = Path(
    "/content/drive/MyDrive/vn_history_model_backups/"
    "vn_history_rag_sft_dataset/all_messages.jsonl"
)

messages = read_jsonl(MESSAGES_PATH)

QUESTION_RE = re.compile(r'Câu hỏi:\s*(.*?)(?:\n\s*\n\s*Tài liệu tham khảo:|$)', re.I | re.S)
SOURCE_LINE_RE = re.compile(r'Nguồn được dùng:\s*\[(.*?)\]', re.I | re.S)


def parse_message_sample(sample: Dict[str, Any], idx: int) -> Optional[Dict[str, Any]]:
    user_text, assistant_text = '', ''
    for m in sample.get('messages', []):
        if m.get('role') == 'user':
            user_text = m.get('content', '')
        elif m.get('role') == 'assistant':
            assistant_text = m.get('content', '')
    if not user_text or not assistant_text:
        return None

    qm = QUESTION_RE.search(user_text)
    question = clean_text(qm.group(1) if qm else user_text)

    sm = SOURCE_LINE_RE.search(assistant_text)
    gold_ids = []
    if sm:
        inside = sm.group(1).strip()
        if inside:
            gold_ids = [x.strip().strip('"\' ') for x in re.split(r'[,;]+', inside) if x.strip()]

    ans_parts = ANSWER_SPLIT_RE.split(assistant_text, maxsplit=1)
    answer_body = clean_text(ans_parts[1] if len(ans_parts) > 1 else assistant_text)

    return {
        'id': sample.get('id', f'sample_{idx:04d}'),
        'type': sample.get('type', 'unknown'),
        'question': question,
        'reference_answer': answer_body,
        'gold_source_ids': gold_ids,
    }

records = [r for i, s in enumerate(messages) if (r := parse_message_sample(s, i)) is not None]
df = pd.DataFrame(records)

# Tái tạo chính xác split Phase2: 90/5/5, stratified, seed 42.
train_df, temp_df = train_test_split(
    df,
    test_size=0.10,
    random_state=SEED,
    stratify=df['type'],
)
eval_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_df['type'],
)
heldout = pd.concat([eval_df, test_df], ignore_index=True)

print('Reconstructed train/eval/test:', len(train_df), len(eval_df), len(test_df))
print('Held-out type counts:')
print(heldout['type'].value_counts())

history_n = min(BENCHMARK_HISTORY_N, len(heldout))
history_bench = heldout.sample(n=history_n, random_state=SEED).copy()
history_bench['is_off_topic'] = False
history_bench['expected_refusal'] = history_bench['type'].isin(['insufficient_context', 'false_premise'])

OOD_QUESTIONS = [
    'Viết cho tôi một hàm Python để merge hai dictionary.',
    'Thời tiết Thành phố Hồ Chí Minh hôm nay có mưa không?',
    'Messi đã ghi bao nhiêu bàn ở mùa giải gần nhất?',
    'Cách nấu bò kho ngon tại nhà như thế nào?',
    'Giải phương trình x^2 - 5x + 6 = 0.',
    'Đau đầu và sốt nhẹ thì nên uống thuốc gì?',
    'Giá Bitcoin hôm nay là bao nhiêu?',
    'Tôi nên làm gì khi người yêu không trả lời tin nhắn?',
    'Dịch câu "machine learning is useful" sang tiếng Việt.',
    'So sánh iPhone và Samsung đời mới nhất.',
][:BENCHMARK_OOD_N]

ood_rows = []
for i, q in enumerate(OOD_QUESTIONS, 1):
    ood_rows.append({
        'id': f'ood_{i:02d}',
        'type': 'off_topic',
        'question': q,
        'reference_answer': '',
        'gold_source_ids': [],
        'is_off_topic': True,
        'expected_refusal': True,
    })

benchmark_df = pd.concat([history_bench, pd.DataFrame(ood_rows)], ignore_index=True)
benchmark_df = benchmark_df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)

# Mark gold IDs còn tồn tại trong corpus Phase8 hiện tại.
benchmark_df['gold_source_ids_present'] = benchmark_df['gold_source_ids'].apply(
    lambda xs: [x for x in (xs or []) if x in CORPUS_ID_SET]
)
benchmark_df['missing_gold_ids'] = benchmark_df.apply(
    lambda r: [x for x in (r['gold_source_ids'] or []) if x not in CORPUS_ID_SET], axis=1
)

with BENCHMARK_PATH.open('w', encoding='utf-8') as f:
    for row in benchmark_df.to_dict('records'):
        f.write(json.dumps(row, ensure_ascii=False) + '\n')

print('Benchmark size:', len(benchmark_df))
print('Off-topic:', int(benchmark_df['is_off_topic'].sum()))
print('Expected refusal:', int(benchmark_df['expected_refusal'].sum()))
print('Rows with missing old gold IDs:', int(benchmark_df['missing_gold_ids'].apply(bool).sum()))
print('Saved:', BENCHMARK_PATH)
display(benchmark_df[['id','type','question','expected_refusal']].head(12))

Reconstructed train/eval/test: 900 50 50
Held-out type counts:
type
noisy_context           65
grounded_qa             20
insufficient_context    10
false_premise            5
Name: count, dtype: int64
Benchmark size: 100
Off-topic: 10
Expected refusal: 22
Rows with missing old gold IDs: 2
Saved: /content/drive/MyDrive/vn_history_model_backups/rag_corpus_vn_history/phase9_hybrid_rag/benchmark_100.jsonl


,id,type,question,expected_refusal
0,sample_0018,noisy_context,Phùng Hưng đánh phủ đô hộ và Cao Chính Bình th...,False
1,sample_0032,grounded_qa,Phan Châu Trinh sinh và mất vào thời gian nào?,False
2,sample_0016,grounded_qa,"Tháng 4 âm lịch năm 1289, Trần Hưng Đạo được p...",False
3,sample_0005,noisy_context,Khoa thi đầu tiên dưới thời Lý được mở khi nào...,False
4,sample_0007,noisy_context,Đoàn 559 được thành lập trong bối cảnh nào năm...,False
5,sample_0001,noisy_context,Nhà Lý là triều đại nào trong lịch sử Việt Nam?,False
6,sample_0013,noisy_context,Trần Hưng Đạo nêu thượng sách giữ nước là gì t...,False
7,sample_0003,noisy_context,Nguồn gốc và tên gọi ban đầu của Hồ Quý Ly đượ...,False
8,sample_0030,grounded_qa,Đề Nắm bị giết vào thời điểm nào?,False
9,sample_0033,grounded_qa,Thành Hoa Lư có diện tích hơn bao nhiêu hecta?,False


## 15. Định nghĩa evaluation metrics

Các nhóm metric:

### Chất lượng câu trả lời
- semantic similarity;
- Token F1;
- ROUGE-L;
- Year F1.

### Grounding / chống bịa
- source precision / recall / F1;
- source validity;
- unsupported-year-free;
- grounding support.

### Retrieval
- Recall@20;
- MRR@20;
- final-context recall.

### Hành vi
- refusal accuracy;
- off-topic refusal accuracy;
- behavior accuracy.

Không nên đánh giá hệ thống chỉ bằng một metric duy nhất.


In [15]:
# Cell 15 — Evaluation metrics

def metric_tokens(text: str) -> List[str]:
    return re.findall(r"[0-9A-Za-zÀ-ỹĐđ]+", clean_text(text).lower())


def token_f1(pred: str, ref: str) -> float:
    p, r = metric_tokens(pred), metric_tokens(ref)
    if not p and not r:
        return 1.0
    if not p or not r:
        return 0.0
    cp, cr = Counter(p), Counter(r)
    common = sum((cp & cr).values())
    precision = common / len(p)
    recall = common / len(r)
    return 0.0 if precision + recall == 0 else 2 * precision * recall / (precision + recall)


def rouge_l_f1(pred: str, ref: str) -> float:
    a, b = metric_tokens(pred), metric_tokens(ref)
    if not a and not b:
        return 1.0
    if not a or not b:
        return 0.0
    # LCS with rolling DP.
    if len(b) > len(a):
        a, b = b, a
    prev = [0] * (len(b) + 1)
    for x in a:
        cur = [0]
        for j, y in enumerate(b, 1):
            cur.append(prev[j-1] + 1 if x == y else max(prev[j], cur[-1]))
        prev = cur
    lcs = prev[-1]
    p = lcs / len(a)
    r = lcs / len(b)
    return 0.0 if p + r == 0 else 2 * p * r / (p + r)


def year_f1(pred: str, ref: str) -> float:
    p, r = extract_year_set(pred), extract_year_set(ref)
    if not p and not r:
        return 1.0
    if not p or not r:
        return 0.0
    common = len(p & r)
    pr = common / len(p)
    rc = common / len(r)
    return 0.0 if pr + rc == 0 else 2 * pr * rc / (pr + rc)


def source_prf(pred_ids: List[str], gold_ids: List[str]) -> Tuple[float,float,float]:
    p, g = set(pred_ids or []), set(gold_ids or [])
    if not p and not g:
        return 1.0, 1.0, 1.0
    precision = len(p & g) / len(p) if p else 0.0
    recall = len(p & g) / len(g) if g else (1.0 if not p else 0.0)
    f1 = 0.0 if precision + recall == 0 else 2 * precision * recall / (precision + recall)
    return precision, recall, f1


def retrieval_metrics(candidate20: List[str], final_ids: List[str], gold_ids: List[str]) -> Dict[str, float]:
    gold = [x for x in (gold_ids or []) if x in CORPUS_ID_SET]
    if not gold:
        return {'retrieval_recall20': np.nan, 'retrieval_recall_final': np.nan, 'mrr20': np.nan}
    gs = set(gold)
    recall20 = len(gs & set(candidate20)) / len(gs)
    recall_final = len(gs & set(final_ids)) / len(gs)
    rr = 0.0
    for rank, cid in enumerate(candidate20, 1):
        if cid in gs:
            rr = 1.0 / rank
            break
    return {'retrieval_recall20': recall20, 'retrieval_recall_final': recall_final, 'mrr20': rr}


def semantic_similarity_batch(preds: List[str], refs: List[str]) -> List[float]:
    if not preds:
        return []
    pe = embedder.encode(['query: ' + clean_text(x) for x in preds], batch_size=64, convert_to_numpy=True, normalize_embeddings=True)
    re_ = embedder.encode(['passage: ' + clean_text(x) for x in refs], batch_size=64, convert_to_numpy=True, normalize_embeddings=True)
    return np.sum(pe * re_, axis=1).astype(float).tolist()

def benchmark_quality_diagnostics(question: str, answer: str) -> Dict[str,Any]:
    crit=critique_answer_core(question,answer)
    n=match_norm(answer)
    return {
        'quality_pass':bool(crit.get('pass',False)),
        'quality_issue_count':len(crit.get('issues',[])),
        'style_spam':bool(re.match(r'\s*(theo|dua tren|can cu).*tai lieu',n) or re.search(r'\bchunk\b',n)),
    }


## 16. Benchmark 4 cấu hình — v2 checkpoint riêng

So sánh:

1. **Vanilla Qwen2.5-3B-Instruct**
2. **Stage 1 merged**
3. **Stage 1 + Stage 2 weights-only**
4. **Stage 1 + Stage 2 + Full Hybrid RAG**

### Lưu ý tài nguyên

Đây là cell nặng nhất notebook.

- 100 câu × 4 cấu hình ≈ **400 lượt generation**.
- **A100 40/80 GB khuyến nghị mạnh**.
- L4 có thể chạy nhưng lâu hơn.
- T4 dễ thiếu VRAM và thời gian chạy dài.

Kết quả được append định kỳ vào checkpoint; nếu runtime ngắt, chạy lại cell sẽ bỏ qua các sample đã hoàn tất.


### Benchmark v2 dùng checkpoint riêng

- `benchmark_results_v2_tooluse.jsonl`
- `benchmark_summary_v2_tooluse.csv`

Không trộn với benchmark cũ đã bị dừng. FAISS/BM25 cache vẫn tái sử dụng bình thường.


In [17]:
# ============================================================
# FIX — helper bị thiếu cho batched Full RAG benchmark
# ============================================================

def flatten_full_rag_result(result: Dict[str, Any]) -> Dict[str, Any]:
    retr = result.get("retrieval", {})

    return {
        "answer":
            result.get("answer", ""),

        "raw_output":
            result.get("raw_output", ""),

        "source_ids":
            result.get("source_ids", []),

        "format_ok":
            result.get("format_ok", False),

        "status":
            result.get("status", ""),

        "invalid_source_ids":
            result.get("invalid_source_ids", []),

        "unsupported_years":
            result.get("unsupported_years", []),

        "support_score":
            result.get("support_score"),

        "candidate20_ids": [
            str(c["chunk_id"])
            for c in retr.get(
                "candidates20",
                []
            )
        ],

        "final_context_ids": [
            str(c["chunk_id"])
            for c in retr.get(
                "final_context",
                []
            )
        ],

        "rewrite_used":
            bool(
                result.get(
                    "rewrite_used",
                    False
                )
            ),

        "quality_warnings":
            result.get(
                "quality_warnings",
                []
            ),

        "context_title_diversity":
            retr.get(
                "context_title_diversity",
                np.nan
            ),

        "query_variant_count":
            len(
                retr.get(
                    "query_variants",
                    []
                )
            ),

        "latency_sec":
            result.get(
                "latency_sec"
            ),
    }


print("✅ flatten_full_rag_result ready")

✅ flatten_full_rag_result ready


In [20]:
# ============================================================
# Cell 16 — FAST BATCHED BENCHMARK v3
# ============================================================
#
# 1) vanilla
# 2) stage1
# 3) stage12_weights
# 4) stage12_full_rag
#
# FIX QUAN TRỌNG:
# - benchmark_uid UNIQUE cho từng row
# - checkpoint dùng benchmark_uid thay vì id gốc
# - generation thật sự theo batch
# - Full RAG batch initial generation + batch repair
#
# Yêu cầu Cell 2:
# GENERATION_BATCH_SIZE = 16
# ============================================================


# ============================================================
# 0. Dùng checkpoint MỚI
# Không trộn với benchmark_results_v2_tooluse.jsonl cũ
# ============================================================

BENCHMARK_RESULTS_PATH = (
    PHASE9_DIR
    / "benchmark_results_v3_unique_batched.jsonl"
)

BENCHMARK_SUMMARY_PATH = (
    PHASE9_DIR
    / "benchmark_summary_v3_unique_batched.csv"
)

print("Benchmark result path:")
print(BENCHMARK_RESULTS_PATH)


# ============================================================
# 1. Build 100 rows + UNIQUE benchmark_uid
# ============================================================

benchmark_rows = benchmark_df.to_dict(
    "records"
)

for idx, row in enumerate(
    benchmark_rows
):

    original_id = str(
        row.get(
            "id",
            "sample"
        )
    )

    question = clean_text(
        row.get(
            "question",
            ""
        )
    )

    row_type = str(
        row.get(
            "type",
            ""
        )
    )

    # Stable hash cho nội dung row
    signature = (
        original_id
        + "||"
        + row_type
        + "||"
        + question
    )

    short_hash = hashlib.sha1(
        signature.encode(
            "utf-8"
        )
    ).hexdigest()[:12]

    # idx đảm bảo UNIQUE ngay cả khi dataset
    # vô tình có hai row giống hệt nhau.
    row[
        "benchmark_uid"
    ] = (
        f"{idx:04d}_"
        f"{original_id}_"
        f"{short_hash}"
    )


uids = [
    row[
        "benchmark_uid"
    ]
    for row
    in benchmark_rows
]

print(
    "Benchmark rows:",
    len(benchmark_rows)
)

print(
    "Unique benchmark_uid:",
    len(set(uids))
)

assert (
    len(benchmark_rows)
    ==
    len(set(uids))
), (
    "benchmark_uid không unique!"
)

print(
    "✅ benchmark_uid unique hoàn toàn."
)


# ============================================================
# 2. Checkpoint helpers
# ============================================================

def load_benchmark_done(
) -> Dict[
    Tuple[str, str],
    Dict[str, Any]
]:

    if not BENCHMARK_RESULTS_PATH.exists():
        return {}

    out = {}

    for r in read_jsonl(
        BENCHMARK_RESULTS_PATH
    ):

        uid = r.get(
            "benchmark_uid"
        )

        variant = r.get(
            "variant"
        )

        if not uid or not variant:
            continue

        out[
            (
                str(variant),
                str(uid),
            )
        ] = r

    return out


def save_benchmark_buffer(
    buffer
):

    if not buffer:
        return

    append_jsonl(
        BENCHMARK_RESULTS_PATH,
        buffer,
    )


# ============================================================
# 3. TRUE BATCH GENERATION
# ============================================================

@torch.inference_mode()
def _generate_raw_batch_once(
    model,
    tok,
    prompts: List[str],
    max_new_tokens: int,
) -> List[str]:

    if not prompts:
        return []

    device = get_model_device(
        model
    )

    inputs = tok(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_INPUT_TOKENS,
        add_special_tokens=False,
    ).to(device)

    # tokenizer đang left-padding
    input_width = (
        inputs[
            "input_ids"
        ].shape[1]
    )

    im_end_id = (
        tok.convert_tokens_to_ids(
            IM_END
        )
    )

    eos_ids = [
        tok.eos_token_id
    ]

    if (
        isinstance(
            im_end_id,
            int
        )
        and
        im_end_id >= 0
    ):

        eos_ids.append(
            im_end_id
        )

    kwargs = dict(
        **inputs,

        max_new_tokens=
            max_new_tokens,

        do_sample=
            TEMPERATURE > 0,

        repetition_penalty=
            REPETITION_PENALTY,

        eos_token_id=
            eos_ids,

        pad_token_id=
            tok.pad_token_id,
    )

    if TEMPERATURE > 0:

        kwargs[
            "temperature"
        ] = TEMPERATURE

        kwargs[
            "top_p"
        ] = TOP_P


    outputs = model.generate(
        **kwargs
    )


    raw_outputs = []

    for i in range(
        len(prompts)
    ):

        generated = outputs[
            i,
            input_width:
        ]

        raw_outputs.append(
            tok.decode(
                generated,
                skip_special_tokens=False,
            )
        )


    del inputs
    del outputs

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return raw_outputs


# ============================================================
# 4. Adaptive OOM fallback
# ============================================================

def generate_raw_batch(
    model,
    tok,
    prompts: List[str],
    max_new_tokens: int,
) -> List[str]:

    if not prompts:
        return []

    try:

        return (
            _generate_raw_batch_once(
                model,
                tok,
                prompts,
                max_new_tokens,
            )
        )

    except RuntimeError as exc:

        msg = str(
            exc
        ).lower()

        is_oom = (
            "out of memory"
            in msg
            or
            "cuda error"
            in msg
        )

        if not is_oom:
            raise

        print(
            f"\n⚠️ CUDA OOM "
            f"batch={len(prompts)}"
        )

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        gc.collect()

        if len(prompts) <= 1:
            raise

        mid = (
            len(prompts)
            // 2
        )

        left = prompts[
            :mid
        ]

        right = prompts[
            mid:
        ]

        print(
            "→ split:",
            len(left),
            "+",
            len(right),
        )

        return (
            generate_raw_batch(
                model,
                tok,
                left,
                max_new_tokens,
            )
            +
            generate_raw_batch(
                model,
                tok,
                right,
                max_new_tokens,
            )
        )


# ============================================================
# 5. Vanilla / Stage1 batch
# ============================================================

def run_plain_batch(
    model,
    tok,
    rows,
    variant,
):

    prompts = [
        build_plain_prompt(
            row[
                "question"
            ]
        )
        for row
        in rows
    ]

    started = (
        time.perf_counter()
    )

    raws = generate_raw_batch(
        model,
        tok,
        prompts,
        max_new_tokens=
            BENCHMARK_MAX_NEW_TOKENS,
    )

    wall = (
        time.perf_counter()
        - started
    )

    avg_latency = (
        wall
        /
        max(
            1,
            len(rows)
        )
    )

    outputs = []

    for raw in raws:

        answer = (
            extract_plain_answer(
                raw,
                tok,
            )
        )

        outputs.append({

            "answer":
                answer,

            "raw_output":
                clean_generated(
                    raw,
                    tok,
                ),

            "source_ids":
                [],

            "format_ok":
                None,

            "status":
                "plain_generation",

            "invalid_source_ids":
                [],

            "unsupported_years":
                [],

            "support_score":
                None,

            "candidate20_ids":
                [],

            "final_context_ids":
                [],

            "rewrite_used":
                False,

            "quality_warnings":
                [],

            "context_title_diversity":
                np.nan,

            "query_variant_count":
                0,

            "latency_sec":
                avg_latency,

            "batch_wall_sec":
                wall,

            "batch_size_used":
                len(rows),
        })

    return outputs


# ============================================================
# 6. Stage12 weights-only batch
# ============================================================

def run_stage12_weights_batch(
    model,
    tok,
    rows,
):

    prompts = []

    for row in rows:

        user_text = (
            build_rag_user_text(
                row[
                    "question"
                ],
                [],
                0,
            )
        )

        prompts.append(
            build_rag_prompt(
                user_text
            )
        )


    started = (
        time.perf_counter()
    )

    raws = generate_raw_batch(
        model,
        tok,
        prompts,
        max_new_tokens=
            BENCHMARK_MAX_NEW_TOKENS,
    )

    wall = (
        time.perf_counter()
        - started
    )

    avg_latency = (
        wall
        /
        max(
            1,
            len(rows)
        )
    )


    outputs = []

    for raw in raws:

        parsed = (
            parse_rag_output(
                raw,
                tok,
            )
        )

        outputs.append({

            "answer":
                parsed[
                    "answer"
                ],

            "raw_output":
                parsed[
                    "raw_output"
                ],

            "source_ids":
                parsed[
                    "source_ids"
                ],

            "format_ok":
                parsed[
                    "format_ok"
                ],

            "status":
                "stage12_no_context",

            # Không có evidence context,
            # citation model tạo ra không thể validate.
            "invalid_source_ids":
                parsed[
                    "source_ids"
                ],

            "unsupported_years":
                list(
                    extract_year_set(
                        parsed[
                            "answer"
                        ]
                    )
                ),

            "support_score":
                None,

            "candidate20_ids":
                [],

            "final_context_ids":
                [],

            "rewrite_used":
                False,

            "quality_warnings":
                [],

            "context_title_diversity":
                np.nan,

            "query_variant_count":
                0,

            "latency_sec":
                avg_latency,

            "batch_wall_sec":
                wall,

            "batch_size_used":
                len(rows),
        })

    return outputs


# ============================================================
# 7. Flatten Full RAG
# ============================================================

def flatten_full_rag_result(
    result: Dict[
        str,
        Any
    ]
) -> Dict[
    str,
    Any
]:

    retr = result.get(
        "retrieval",
        {},
    )

    return {

        "answer":
            result.get(
                "answer",
                "",
            ),

        "raw_output":
            result.get(
                "raw_output",
                "",
            ),

        "source_ids":
            result.get(
                "source_ids",
                [],
            ),

        "format_ok":
            result.get(
                "format_ok",
                False,
            ),

        "status":
            result.get(
                "status",
                "",
            ),

        "invalid_source_ids":
            result.get(
                "invalid_source_ids",
                [],
            ),

        "unsupported_years":
            result.get(
                "unsupported_years",
                [],
            ),

        "support_score":
            result.get(
                "support_score"
            ),

        "candidate20_ids": [
            str(
                c[
                    "chunk_id"
                ]
            )
            for c
            in retr.get(
                "candidates20",
                [],
            )
        ],

        "final_context_ids": [
            str(
                c[
                    "chunk_id"
                ]
            )
            for c
            in retr.get(
                "final_context",
                [],
            )
        ],

        "rewrite_used":
            bool(
                result.get(
                    "rewrite_used",
                    False,
                )
            ),

        "quality_warnings":
            result.get(
                "quality_warnings",
                [],
            ),

        "context_title_diversity":
            retr.get(
                "context_title_diversity",
                np.nan,
            ),

        "query_variant_count":
            len(
                retr.get(
                    "query_variants",
                    [],
                )
            ),

        "latency_sec":
            result.get(
                "latency_sec"
            ),
    }


# ============================================================
# 8. Prepare one Full RAG sample
# ============================================================

def prepare_full_rag_sample(
    row
):

    question = clean_text(
        row[
            "question"
        ]
    )

    analysis = (
        history_question_analyzer.invoke({
            "question":
                question
        })
    )

    query_variants = (
        retrieval_query_planner.invoke({
            "question":
                question
        })
    )

    retrieval = hybrid_retrieve(
        question,
        analysis=analysis,
        query_variants=
            query_variants,
    )


    # ----------------------------------------
    # OOD
    # ----------------------------------------

    if retrieval.get(
        "is_ood"
    ):

        return {

            "ready":
                False,

            "final_result": {

                "question":
                    question,

                "answer":
                    SAFE_OOD_ANSWER,

                "status":
                    "blocked_off_topic",

                "source_ids":
                    [],

                "invalid_source_ids":
                    [],

                "unsupported_years":
                    [],

                "format_ok":
                    True,

                "retrieval":
                    retrieval,

                "analysis":
                    analysis,

                "support_score":
                    None,

                "quality_warnings":
                    [],

                "rewrite_used":
                    False,

                "repair_attempted":
                    False,

                "raw_output":
                    "",

                "tool_trace":
                    retrieval.get(
                        "tool_trace",
                        [],
                    )
                    +
                    [
                        "final_ood_guard"
                    ],
            },
        }


    contexts = retrieval.get(
        "final_context",
        [],
    )


    # ----------------------------------------
    # No context
    # ----------------------------------------

    if not contexts:

        return {

            "ready":
                False,

            "final_result": {

                "question":
                    question,

                "answer":
                    SAFE_INSUFFICIENT_ANSWER,

                "status":
                    "blocked_no_context",

                "source_ids":
                    [],

                "invalid_source_ids":
                    [],

                "unsupported_years":
                    [],

                "format_ok":
                    True,

                "retrieval":
                    retrieval,

                "analysis":
                    analysis,

                "support_score":
                    None,

                "quality_warnings":
                    [],

                "rewrite_used":
                    False,

                "repair_attempted":
                    False,

                "raw_output":
                    "",

                "tool_trace":
                    retrieval.get(
                        "tool_trace",
                        [],
                    )
                    +
                    [
                        "no_context_guard"
                    ],
            },
        }


    prompt, used_context, budget = (
        fit_rag_prompt(
            generation_tokenizer,
            question,
            contexts,
        )
    )

    return {

        "ready":
            True,

        "question":
            question,

        "analysis":
            analysis,

        "retrieval":
            retrieval,

        "contexts":
            contexts,

        "used_context":
            used_context,

        "prompt":
            prompt,

        "budget":
            budget,
    }


# ============================================================
# 9. Full RAG REAL batch
# ============================================================

def run_full_rag_batch(
    rows
):

    batch_started = (
        time.perf_counter()
    )

    prepared = [
        prepare_full_rag_sample(
            row
        )
        for row
        in rows
    ]

    results = [
        None
    ] * len(rows)


    # ========================================================
    # Initial generation indices
    # ========================================================

    gen_indices = [
        idx
        for idx, item
        in enumerate(
            prepared
        )
        if item.get(
            "ready"
        )
    ]


    initial_states = {}


    if gen_indices:

        prompts = [
            prepared[
                idx
            ][
                "prompt"
            ]
            for idx
            in gen_indices
        ]

        raws = (
            generate_raw_batch(
                generation_model,
                generation_tokenizer,
                prompts,

                max_new_tokens=
                    BENCHMARK_MAX_NEW_TOKENS,
            )
        )


        for idx, raw in zip(
            gen_indices,
            raws,
        ):

            item = prepared[
                idx
            ]

            parsed = (
                parse_rag_output(
                    raw,
                    generation_tokenizer,
                )
            )

            validated = (
                validate_parsed_answer(
                    parsed,
                    item[
                        "used_context"
                    ],
                )
            )

            critique = (
                answer_quality_critic.invoke({

                    "question":
                        item[
                            "question"
                        ],

                    "answer":
                        validated[
                            "answer"
                        ],
                })
            )

            reasons = list(
                dict.fromkeys(
                    validated[
                        "guard_issues"
                    ]
                    +
                    critique.get(
                        "issues",
                        [],
                    )
                )
            )

            initial_states[
                idx
            ] = {

                "parsed":
                    parsed,

                "validated":
                    validated,

                "critique":
                    critique,

                "repair_reasons":
                    reasons,

                "budget":
                    item[
                        "budget"
                    ],
            }


    # ========================================================
    # Collect repair prompts
    # ========================================================

    repair_indices = []
    repair_prompts = []
    repair_contexts = {}
    repair_budgets = {}


    for idx in gen_indices:

        state = initial_states[
            idx
        ]

        reasons = state[
            "repair_reasons"
        ]

        answer = state[
            "validated"
        ][
            "answer"
        ]

        should_repair = (
            ENABLE_COMPLETENESS_REWRITE
            and
            bool(reasons)
            and
            MAX_REWRITE_ATTEMPTS > 0
            and
            not is_refusal(
                answer
            )
        )

        if not should_repair:
            continue


        item = prepared[
            idx
        ]

        (
            rewrite_prompt,
            rewrite_context,
            rewrite_budget,
        ) = fit_rewrite_prompt(

            generation_tokenizer,

            item[
                "question"
            ],

            item[
                "contexts"
            ],

            answer,

            reasons,
        )


        repair_indices.append(
            idx
        )

        repair_prompts.append(
            rewrite_prompt
        )

        repair_contexts[
            idx
        ] = rewrite_context

        repair_budgets[
            idx
        ] = rewrite_budget


    # ========================================================
    # Batch repair generation
    # ========================================================

    repaired_states = {}


    if repair_prompts:

        repair_raws = (
            generate_raw_batch(
                generation_model,
                generation_tokenizer,
                repair_prompts,

                max_new_tokens=
                    BENCHMARK_MAX_NEW_TOKENS,
            )
        )


        for idx, raw in zip(
            repair_indices,
            repair_raws,
        ):

            item = prepared[
                idx
            ]

            parsed = (
                parse_rag_output(
                    raw,
                    generation_tokenizer,
                )
            )

            validated = (
                validate_parsed_answer(
                    parsed,
                    repair_contexts[
                        idx
                    ],
                )
            )

            critique = (
                answer_quality_critic.invoke({

                    "question":
                        item[
                            "question"
                        ],

                    "answer":
                        validated[
                            "answer"
                        ],
                })
            )


            repaired_states[
                idx
            ] = {

                "parsed":
                    parsed,

                "validated":
                    validated,

                "critique":
                    critique,

                "budget":
                    repair_budgets[
                        idx
                    ],
            }


    # ========================================================
    # Final decision
    # ========================================================

    for idx, item in enumerate(
        prepared
    ):

        # blocked before generation
        if not item.get(
            "ready"
        ):

            results[
                idx
            ] = item[
                "final_result"
            ]

            continue


        first = initial_states[
            idx
        ]

        chosen = first

        rewrite_used = False

        repair_attempted = (
            idx
            in repair_indices
        )


        # ------------------------------------
        # compare repaired vs initial
        # ------------------------------------

        if idx in repaired_states:

            repaired = (
                repaired_states[
                    idx
                ]
            )

            fv = first[
                "validated"
            ]

            fc = first[
                "critique"
            ]

            rv = repaired[
                "validated"
            ]

            rc = repaired[
                "critique"
            ]


            structurally_better = (
                not rv[
                    "guard_issues"
                ]
            )

            quality_better = (
                len(
                    rc.get(
                        "issues",
                        [],
                    )
                )
                <
                len(
                    fc.get(
                        "issues",
                        [],
                    )
                )
            )

            initial_guard_problem = (
                bool(
                    fv[
                        "guard_issues"
                    ]
                )
            )


            if (
                structurally_better
                and
                (
                    quality_better
                    or
                    initial_guard_problem
                )
            ):

                chosen = repaired

                rewrite_used = True


        validated = chosen[
            "validated"
        ]

        critique = chosen[
            "critique"
        ]

        answer = validated[
            "answer"
        ]

        status = "ok"


        # ------------------------------------
        # Final guards
        # ------------------------------------

        if validated[
            "guard_issues"
        ]:

            issues = validated[
                "guard_issues"
            ]

            if (
                "invalid_source_id"
                in issues
            ):

                status = (
                    "blocked_invalid_source_id"
                )

            elif (
                "missing_source"
                in issues
            ):

                status = (
                    "blocked_missing_source"
                )

            elif (
                "unsupported_year"
                in issues
            ):

                status = (
                    "blocked_unsupported_year"
                )

            else:

                status = (
                    "blocked_guard"
                )


            answer = (
                SAFE_INSUFFICIENT_ANSWER
            )

            source_ids = []


        else:

            source_ids = validated[
                "valid_ids"
            ]

            if critique.get(
                "issues"
            ):

                status = (
                    "ok_with_quality_warning"
                )


        # ------------------------------------
        # Grounding support
        # ------------------------------------

        evidence_chunks = [

            chunk_by_id[
                cid
            ]

            for cid
            in source_ids

            if cid
            in chunk_by_id
        ]


        support_score = (

            answer_support_score(
                answer,
                evidence_chunks,
            )

            if evidence_chunks

            else None
        )


        # ------------------------------------
        # Trace
        # ------------------------------------

        trace = (

            item[
                "retrieval"
            ].get(
                "tool_trace",
                [],
            )

            +
            [
                "qwen_generation_batch",
                "style_polisher",
                "source_year_guard",
                "answer_quality_critic",
            ]
        )


        if repair_attempted:

            trace.append(
                "evidence_only_repair_batch"
            )


        if rewrite_used:

            trace.append(
                "repair_accepted"
            )

        elif repair_attempted:

            trace.append(
                "repair_rejected_keep_first"
            )


        results[
            idx
        ] = {

            "question":
                item[
                    "question"
                ],

            "answer":
                answer,

            "status":
                status,

            "source_ids":
                source_ids,

            "model_source_ids":
                chosen[
                    "parsed"
                ].get(
                    "source_ids",
                    [],
                ),

            "invalid_source_ids":
                validated[
                    "invalid_ids"
                ],

            "unsupported_years":
                validated[
                    "unsupported_years"
                ],

            "format_ok":
                chosen[
                    "parsed"
                ].get(
                    "format_ok",
                    False,
                ),

            "raw_output":
                chosen[
                    "parsed"
                ].get(
                    "raw_output",
                    "",
                ),

            "retrieval":
                item[
                    "retrieval"
                ],

            "analysis":
                item[
                    "analysis"
                ],

            "prompt_budget":
                chosen[
                    "budget"
                ],

            "support_score":
                support_score,

            "quality_warnings":
                critique.get(
                    "issues",
                    [],
                ),

            "rewrite_used":
                rewrite_used,

            "repair_attempted":
                repair_attempted,

            "initial_quality_issues":
                first[
                    "critique"
                ].get(
                    "issues",
                    [],
                ),

            "tool_trace":
                trace,
        }


    # ========================================================
    # Batch wall time
    # ========================================================

    wall = (
        time.perf_counter()
        - batch_started
    )

    avg_latency = (
        wall
        /
        max(
            1,
            len(rows)
        )
    )


    final_outputs = []


    for result in results:

        result[
            "latency_sec"
        ] = avg_latency

        flattened = (
            flatten_full_rag_result(
                result
            )
        )

        flattened[
            "batch_wall_sec"
        ] = wall

        flattened[
            "batch_size_used"
        ] = len(rows)

        final_outputs.append(
            flattened
        )


    return final_outputs


# ============================================================
# 10. Main evaluate_variant
# ============================================================

def evaluate_variant(
    variant: str,
    model=None,
    tok=None,
):

    done = (
        load_benchmark_done()
    )

    pending = [

        row

        for row
        in benchmark_rows

        if (
            variant,
            row[
                "benchmark_uid"
            ],
        )
        not in done
    ]


    print(
        f"\n[{variant}] "
        f"pending "
        f"{len(pending)}/"
        f"{len(benchmark_rows)}"
    )


    if not pending:
        return


    buffer = []

    n_batches = math.ceil(
        len(pending)
        /
        GENERATION_BATCH_SIZE
    )


    for start in tqdm(

        range(
            0,
            len(pending),
            GENERATION_BATCH_SIZE,
        ),

        total=
            n_batches,

        desc=
            variant,

        unit=
            "batch",
    ):

        rows = pending[
            start:
            start
            +
            GENERATION_BATCH_SIZE
        ]


        # ------------------------------------
        # Generation
        # ------------------------------------

        if variant in {
            "vanilla",
            "stage1",
        }:

            outs = run_plain_batch(
                model,
                tok,
                rows,
                variant,
            )


        elif variant == (
            "stage12_weights"
        ):

            outs = (
                run_stage12_weights_batch(
                    model,
                    tok,
                    rows,
                )
            )


        elif variant == (
            "stage12_full_rag"
        ):

            outs = (
                run_full_rag_batch(
                    rows
                )
            )


        else:

            raise ValueError(
                variant
            )


        # ------------------------------------
        # Records
        # ------------------------------------

        for row, out in zip(
            rows,
            outs,
        ):

            rec = {

                "variant":
                    variant,

                "benchmark_uid":
                    row[
                        "benchmark_uid"
                    ],

                # giữ original id
                "id":
                    row.get(
                        "id"
                    ),

                "type":
                    row.get(
                        "type",
                        "",
                    ),

                "question":
                    row[
                        "question"
                    ],

                "reference_answer":
                    row.get(
                        "reference_answer",
                        "",
                    ),

                "gold_source_ids":
                    row.get(
                        "gold_source_ids_present",

                        row.get(
                            "gold_source_ids",
                            [],
                        ),
                    ),

                "is_off_topic":
                    bool(
                        row.get(
                            "is_off_topic",
                            False,
                        )
                    ),

                "expected_refusal":
                    bool(
                        row.get(
                            "expected_refusal",
                            False,
                        )
                    ),

                **out,
            }


            buffer.append(
                rec
            )


        # ------------------------------------
        # Save checkpoint mỗi batch
        # ------------------------------------

        if (
            len(buffer)
            >=
            BENCHMARK_SAVE_EVERY
        ):

            save_benchmark_buffer(
                buffer
            )

            buffer = []


        if torch.cuda.is_available():
            torch.cuda.empty_cache()


    save_benchmark_buffer(
        buffer
    )


# ============================================================
# 11. Run all four variants
# ============================================================

print(
    "\nGENERATION_BATCH_SIZE:",
    GENERATION_BATCH_SIZE,
)


# ---------------- VANILLA ----------------

if any(

    (
        "vanilla",
        row[
            "benchmark_uid"
        ],
    )
    not in load_benchmark_done()

    for row
    in benchmark_rows
):

    print(
        "\nLoading VANILLA..."
    )

    m, t = load_model_variant(
        "vanilla"
    )

    evaluate_variant(
        "vanilla",
        m,
        t,
    )

    del m, t

    cleanup_cuda()

else:

    print(
        "Vanilla already complete."
    )


# ---------------- STAGE 1 ----------------

if any(

    (
        "stage1",
        row[
            "benchmark_uid"
        ],
    )
    not in load_benchmark_done()

    for row
    in benchmark_rows
):

    print(
        "\nLoading STAGE1 merged..."
    )

    m, t = load_model_variant(
        "stage1"
    )

    evaluate_variant(
        "stage1",
        m,
        t,
    )

    del m, t

    cleanup_cuda()

else:

    print(
        "Stage1 already complete."
    )


# ---------------- STAGE12 WEIGHTS ----------------

if any(

    (
        "stage12_weights",
        row[
            "benchmark_uid"
        ],
    )
    not in load_benchmark_done()

    for row
    in benchmark_rows
):

    print(
        "\nRunning "
        "STAGE1+STAGE2 "
        "weights-only..."
    )

    evaluate_variant(
        "stage12_weights",
        generation_model,
        generation_tokenizer,
    )

else:

    print(
        "Stage12 weights "
        "already complete."
    )


# ---------------- FULL RAG ----------------

if any(

    (
        "stage12_full_rag",
        row[
            "benchmark_uid"
        ],
    )
    not in load_benchmark_done()

    for row
    in benchmark_rows
):

    print(
        "\nRunning FULL HYBRID RAG..."
    )

    evaluate_variant(
        "stage12_full_rag"
    )

else:

    print(
        "Full RAG already complete."
    )


# ============================================================
# 12. Final checkpoint validation
# ============================================================

done = load_benchmark_done()

print(
    "\n===== CHECKPOINT COUNTS ====="
)

for variant in [
    "vanilla",
    "stage1",
    "stage12_weights",
    "stage12_full_rag",
]:

    count = sum(
        1
        for key
        in done
        if key[0] == variant
    )

    print(
        f"{variant:22s}: "
        f"{count}/"
        f"{len(benchmark_rows)}"
    )


print(
    "\n✅ Benchmark generations complete:"
)

print(
    BENCHMARK_RESULTS_PATH
)

Benchmark result path:
/content/drive/MyDrive/vn_history_model_backups/rag_corpus_vn_history/phase9_hybrid_rag/benchmark_results_v3_unique_batched.jsonl
Benchmark rows: 100
Unique benchmark_uid: 100
✅ benchmark_uid unique hoàn toàn.

GENERATION_BATCH_SIZE: 16

Loading VANILLA...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]


[vanilla] pending 100/100


vanilla:   0%|          | 0/7 [00:00<?, ?batch/s]


Loading STAGE1 merged...


...

Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]


===== CHECKPOINT COUNTS =====
vanilla               : 100/100
stage1                : 100/100
stage12_weights       : 100/100
stage12_full_rag      : 100/100

✅ Benchmark generations complete:
/content/drive/MyDrive/vn_history_model_backups/rag_corpus_vn_history/phase9_hybrid_rag/benchmark_results_v3_unique_batched.jsonl


## 17. Tổng hợp benchmark

Đọc checkpoint benchmark và tạo bảng so sánh cuối giữa bốn cấu hình.

Hãy ưu tiên đọc đồng thời:

- answer quality;
- behavior accuracy;
- source validity;
- retrieval recall;
- latency.

Một hệ thống chống bịa tốt có thể **từ chối nhiều hơn** nhưng vẫn tốt hơn model trả lời trôi chảy mà không có bằng chứng.


In [21]:
# ============================================================
# Cell 17 — FINAL BENCHMARK METRICS v3
#
# FIX:
# - dedup bằng benchmark_uid
# - validate đúng 100 rows / variant
# - metric không áp dụng → NaN
# - Full RAG metrics riêng
# ============================================================


# ============================================================
# 1. Load results
# ============================================================

results = read_jsonl(
    BENCHMARK_RESULTS_PATH
)

print(
    "Raw result records:",
    len(results)
)


res_df = pd.DataFrame(
    results
)


# ============================================================
# 2. Validate required fields
# ============================================================

required_cols = [
    "variant",
    "benchmark_uid",
    "question",
    "answer",
]

missing_cols = [
    c
    for c
    in required_cols
    if c
    not in res_df.columns
]

if missing_cols:

    raise ValueError(
        "Thiếu columns: "
        + str(
            missing_cols
        )
    )


# ============================================================
# 3. Dedup CORRECTLY
# ============================================================

before = len(
    res_df
)

res_df = (
    res_df
    .drop_duplicates(
        subset=[
            "variant",
            "benchmark_uid",
        ],
        keep="last",
    )
    .reset_index(
        drop=True
    )
)

after = len(
    res_df
)


print(
    "Before dedup:",
    before
)

print(
    "After dedup :",
    after
)


# ============================================================
# 4. Count validation
# ============================================================

expected_n = len(
    benchmark_rows
)

count_table = (
    res_df
    .groupby(
        "variant"
    )[
        "benchmark_uid"
    ]
    .nunique()
    .reindex([
        "vanilla",
        "stage1",
        "stage12_weights",
        "stage12_full_rag",
    ])
)


print(
    "\n===== UNIQUE SAMPLE COUNTS ====="
)

display(
    count_table.to_frame(
        "n_unique"
    )
)


for variant, count in (
    count_table.items()
):

    if pd.isna(
        count
    ):
        print(
            f"❌ {variant}: 0/"
            f"{expected_n}"
        )

    elif int(
        count
    ) != expected_n:

        print(
            f"⚠️ {variant}: "
            f"{int(count)}/"
            f"{expected_n}"
        )

    else:

        print(
            f"✅ {variant}: "
            f"{int(count)}/"
            f"{expected_n}"
        )


# ============================================================
# 5. Behavior / refusal
# ============================================================

res_df[
    "refusal"
] = (
    res_df[
        "answer"
    ].apply(
        is_refusal
    )
)


res_df[
    "behavior_correct"
] = (
    res_df[
        "refusal"
    ]
    ==
    res_df[
        "expected_refusal"
    ]
)


# ============================================================
# 6. Text metrics
# ============================================================

res_df[
    "token_f1"
] = res_df.apply(

    lambda r:

        token_f1(
            r[
                "answer"
            ],
            r[
                "reference_answer"
            ],
        )

        if not r[
            "is_off_topic"
        ]

        else np.nan,

    axis=1,
)


res_df[
    "rougeL_f1"
] = res_df.apply(

    lambda r:

        rouge_l_f1(
            r[
                "answer"
            ],
            r[
                "reference_answer"
            ],
        )

        if not r[
            "is_off_topic"
        ]

        else np.nan,

    axis=1,
)


res_df[
    "year_f1"
] = res_df.apply(

    lambda r:

        year_f1(
            r[
                "answer"
            ],
            r[
                "reference_answer"
            ],
        )

        if not r[
            "is_off_topic"
        ]

        else np.nan,

    axis=1,
)


# ============================================================
# 7. Semantic similarity — batch
# ============================================================

semantic_scores = [
    np.nan
] * len(
    res_df
)


semantic_mask = (
    (~res_df[
        "is_off_topic"
    ])
    &
    res_df[
        "reference_answer"
    ].astype(
        bool
    )
)


semantic_idxs = (
    res_df.index[
        semantic_mask
    ].tolist()
)


if semantic_idxs:

    sims = (
        semantic_similarity_batch(

            res_df.loc[
                semantic_idxs,
                "answer",
            ].tolist(),

            res_df.loc[
                semantic_idxs,
                "reference_answer",
            ].tolist(),
        )
    )


    for idx, sim in zip(
        semantic_idxs,
        sims,
    ):

        semantic_scores[
            idx
        ] = float(
            sim
        )


res_df[
    "semantic_similarity"
] = semantic_scores


# ============================================================
# 8. Source / retrieval metrics
# ============================================================

source_precision = []
source_recall = []
source_f1_values = []

recall20 = []
recall_final = []
mrr20_values = []


for _, r in res_df.iterrows():

    predicted_sources = (
        r.get(
            "source_ids",
            [],
        )
        or []
    )

    gold_sources = (
        r.get(
            "gold_source_ids",
            [],
        )
        or []
    )


    p, recall, f1 = (
        source_prf(
            predicted_sources,
            gold_sources,
        )
    )

    source_precision.append(
        p
    )

    source_recall.append(
        recall
    )

    source_f1_values.append(
        f1
    )


    retrieval_result = (
        retrieval_metrics(

            r.get(
                "candidate20_ids",
                [],
            )
            or [],

            r.get(
                "final_context_ids",
                [],
            )
            or [],

            gold_sources,
        )
    )


    recall20.append(
        retrieval_result[
            "retrieval_recall20"
        ]
    )

    recall_final.append(
        retrieval_result[
            "retrieval_recall_final"
        ]
    )

    mrr20_values.append(
        retrieval_result[
            "mrr20"
        ]
    )


res_df[
    "source_precision"
] = source_precision

res_df[
    "source_recall"
] = source_recall

res_df[
    "source_f1"
] = source_f1_values

res_df[
    "retrieval_recall20"
] = recall20

res_df[
    "retrieval_recall_final"
] = recall_final

res_df[
    "mrr20"
] = mrr20_values


# ============================================================
# 9. Guard metrics
# ============================================================

res_df[
    "source_validity"
] = (
    res_df[
        "invalid_source_ids"
    ].apply(

        lambda xs:
            1.0
            if not (
                xs
                or []
            )
            else 0.0
    )
)


res_df[
    "unsupported_year_free"
] = (
    res_df[
        "unsupported_years"
    ].apply(

        lambda xs:
            1.0
            if not (
                xs
                or []
            )
            else 0.0
    )
)


# ============================================================
# 10. Quality / style diagnostics
# ============================================================

quality_results = res_df.apply(

    lambda r:

        benchmark_quality_diagnostics(
            r[
                "question"
            ],
            r[
                "answer"
            ],
        ),

    axis=1,
)


res_df[
    "quality_pass"
] = [
    x[
        "quality_pass"
    ]
    for x
    in quality_results
]


res_df[
    "quality_issue_count"
] = [
    x[
        "quality_issue_count"
    ]
    for x
    in quality_results
]


res_df[
    "style_spam"
] = [
    x[
        "style_spam"
    ]
    for x
    in quality_results
]


# ============================================================
# 11. Ensure optional columns
# ============================================================

if (
    "rewrite_used"
    not in res_df.columns
):

    res_df[
        "rewrite_used"
    ] = False


if (
    "context_title_diversity"
    not in res_df.columns
):

    res_df[
        "context_title_diversity"
    ] = np.nan


if (
    "query_variant_count"
    not in res_df.columns
):

    res_df[
        "query_variant_count"
    ] = 0


# ============================================================
# 12. Safe mean
# ============================================================

def nanmean(
    series
):

    x = pd.to_numeric(
        series,
        errors="coerce",
    )

    if not x.notna().any():
        return np.nan

    return float(
        x.mean()
    )


# ============================================================
# 13. Build FINAL summary
# ============================================================

summary_rows = []


variant_order = [
    "vanilla",
    "stage1",
    "stage12_weights",
    "stage12_full_rag",
]


for variant in variant_order:

    g = res_df[
        res_df[
            "variant"
        ]
        ==
        variant
    ].copy()


    if len(
        g
    ) == 0:
        continue


    hist = g[
        ~g[
            "is_off_topic"
        ]
    ]

    ood = g[
        g[
            "is_off_topic"
        ]
    ]


    gold_mask = (
        g[
            "gold_source_ids"
        ].apply(
            bool
        )
    )


    is_full_rag = (
        variant
        ==
        "stage12_full_rag"
    )


    summary_rows.append({

        "variant":
            variant,

        "n":
            len(g),

        # ------------------------------------
        # Answer quality
        # ------------------------------------

        "semantic_similarity":
            nanmean(
                hist[
                    "semantic_similarity"
                ]
            ),

        "token_f1":
            nanmean(
                hist[
                    "token_f1"
                ]
            ),

        "rougeL_f1":
            nanmean(
                hist[
                    "rougeL_f1"
                ]
            ),

        "year_f1":
            nanmean(
                hist[
                    "year_f1"
                ]
            ),


        # ------------------------------------
        # Behavior
        # ------------------------------------

        "behavior_accuracy_all":
            nanmean(
                g[
                    "behavior_correct"
                ]
            ),

        "off_topic_refusal_accuracy":
            (
                nanmean(
                    ood[
                        "behavior_correct"
                    ]
                )
                if len(
                    ood
                )
                else np.nan
            ),

        "history_behavior_accuracy":
            (
                nanmean(
                    hist[
                        "behavior_correct"
                    ]
                )
                if len(
                    hist
                )
                else np.nan
            ),


        # ------------------------------------
        # Quality/style
        # ------------------------------------

        "answer_quality_pass_rate":
            (
                nanmean(
                    hist[
                        "quality_pass"
                    ]
                )
                if len(
                    hist
                )
                else np.nan
            ),

        "style_spam_rate":
            (
                nanmean(
                    hist[
                        "style_spam"
                    ]
                )
                if len(
                    hist
                )
                else np.nan
            ),


        # ------------------------------------
        # Structured format
        #
        # Vanilla / Stage1 không dùng
        # structured RAG format nên NaN hợp lý.
        # ------------------------------------

        "format_rate":
            (
                nanmean(
                    g[
                        "format_ok"
                    ]
                )
                if variant in {
                    "stage12_weights",
                    "stage12_full_rag",
                }
                else np.nan
            ),


        # ------------------------------------
        # Grounding / source metrics
        #
        # CHỈ full RAG có evidence thực.
        # ------------------------------------

        "source_precision":
            (
                nanmean(
                    g[
                        gold_mask
                    ][
                        "source_precision"
                    ]
                )
                if (
                    is_full_rag
                    and
                    gold_mask.any()
                )
                else np.nan
            ),

        "source_recall":
            (
                nanmean(
                    g[
                        gold_mask
                    ][
                        "source_recall"
                    ]
                )
                if (
                    is_full_rag
                    and
                    gold_mask.any()
                )
                else np.nan
            ),

        "source_f1":
            (
                nanmean(
                    g[
                        gold_mask
                    ][
                        "source_f1"
                    ]
                )
                if (
                    is_full_rag
                    and
                    gold_mask.any()
                )
                else np.nan
            ),

        "source_validity":
            (
                nanmean(
                    g[
                        "source_validity"
                    ]
                )
                if is_full_rag
                else np.nan
            ),

        "unsupported_year_free":
            (
                nanmean(
                    g[
                        "unsupported_year_free"
                    ]
                )
                if is_full_rag
                else np.nan
            ),


        # ------------------------------------
        # Retrieval
        # ------------------------------------

        "retrieval_recall20":
            (
                nanmean(
                    g[
                        "retrieval_recall20"
                    ]
                )
                if is_full_rag
                else np.nan
            ),

        "retrieval_recall_final":
            (
                nanmean(
                    g[
                        "retrieval_recall_final"
                    ]
                )
                if is_full_rag
                else np.nan
            ),

        "mrr20":
            (
                nanmean(
                    g[
                        "mrr20"
                    ]
                )
                if is_full_rag
                else np.nan
            ),


        # ------------------------------------
        # Grounding proxy
        # ------------------------------------

        "grounding_support":
            (
                nanmean(
                    g[
                        "support_score"
                    ]
                )
                if is_full_rag
                else np.nan
            ),


        # ------------------------------------
        # v2/v3 RAG diagnostics
        # ------------------------------------

        "context_title_diversity":
            (
                nanmean(
                    g[
                        "context_title_diversity"
                    ]
                )
                if is_full_rag
                else np.nan
            ),

        "rewrite_rate":
            (
                nanmean(
                    g[
                        "rewrite_used"
                    ]
                )
                if is_full_rag
                else np.nan
            ),

        "avg_query_variants":
            (
                nanmean(
                    g[
                        "query_variant_count"
                    ]
                )
                if is_full_rag
                else np.nan
            ),


        # ------------------------------------
        # Speed
        # ------------------------------------

        "avg_latency_sec":
            nanmean(
                g[
                    "latency_sec"
                ]
            ),
    })


summary_df = pd.DataFrame(
    summary_rows
)


# ============================================================
# 14. Save summary
# ============================================================

summary_df.to_csv(
    BENCHMARK_SUMMARY_PATH,
    index=False,
    encoding="utf-8-sig",
)


print(
    "\nFINAL COMPARISON — Phase 9 v3"
)

display(
    summary_df.round(
        4
    )
)

print(
    "\nSaved:"
)

print(
    BENCHMARK_SUMMARY_PATH
)


# ============================================================
# 15. Hard validation: phải là 100/variant
# ============================================================

print(
    "\n===== FINAL SAMPLE VALIDATION ====="
)


all_complete = True


for variant in variant_order:

    n = len(
        res_df[
            res_df[
                "variant"
            ]
            ==
            variant
        ]
    )

    if n == expected_n:

        print(
            f"✅ {variant:22s}: "
            f"{n}/{expected_n}"
        )

    else:

        print(
            f"❌ {variant:22s}: "
            f"{n}/{expected_n}"
        )

        all_complete = False


if all_complete:

    print(
        "\n✅ Benchmark đầy đủ "
        f"{expected_n} câu × "
        f"{len(variant_order)} variants."
    )

else:

    print(
        "\n⚠️ Benchmark chưa đầy đủ. "
        "Không nên dùng summary này "
        "làm kết quả cuối."
    )


# ============================================================
# 16. Full RAG diagnostics
# ============================================================

full = res_df[
    res_df[
        "variant"
    ]
    ==
    "stage12_full_rag"
].copy()


if len(
    full
):

    print(
        "\n===== FULL RAG DIAGNOSTICS ====="
    )


    history_full = full[
        ~full[
            "is_off_topic"
        ]
    ]


    print(
        "Full RAG style spam:",
        int(
            history_full[
                "style_spam"
            ].sum()
        ),
        "/",
        len(
            history_full
        ),
    )


    print(
        "Full RAG quality pass:",
        round(
            float(
                history_full[
                    "quality_pass"
                ].mean()
            ),
            4,
        ),
    )


    print(
        "Full RAG rewrite rate:",
        round(
            float(
                pd.to_numeric(
                    full[
                        "rewrite_used"
                    ],
                    errors="coerce",
                ).mean()
            ),
            4,
        ),
    )


    print(
        "Mean context diversity:",
        round(
            nanmean(
                full[
                    "context_title_diversity"
                ]
            ),
            4,
        ),
    )


    print(
        "Recall@20:",
        round(
            nanmean(
                full[
                    "retrieval_recall20"
                ]
            ),
            4,
        ),
    )


    print(
        "Final context recall:",
        round(
            nanmean(
                full[
                    "retrieval_recall_final"
                ]
            ),
            4,
        ),
    )


    print(
        "MRR@20:",
        round(
            nanmean(
                full[
                    "mrr20"
                ]
            ),
            4,
        ),
    )


    print(
        "Grounding support:",
        round(
            nanmean(
                full[
                    "support_score"
                ]
            ),
            4,
        ),
    )


    # ========================================================
    # Behavior errors
    # ========================================================

    behavior_errors = full[
        ~full[
            "behavior_correct"
        ]
    ].copy()


    print(
        "\nFull RAG behavior errors:",
        len(
            behavior_errors
        ),
    )


    if len(
        behavior_errors
    ):

        display(
            behavior_errors[
                [
                    "benchmark_uid",
                    "id",
                    "type",
                    "question",
                    "answer",
                    "status",
                    "gold_source_ids",
                    "source_ids",
                ]
            ].head(
                30
            )
        )


    # ========================================================
    # Quality warnings
    # ========================================================

    quality_bad = full[
        ~full[
            "quality_pass"
        ]
    ].copy()


    print(
        "\nFull RAG quality failures:",
        len(
            quality_bad
        ),
    )


    if len(
        quality_bad
    ):

        display(
            quality_bad[
                [
                    "benchmark_uid",
                    "question",
                    "answer",
                    "quality_issue_count",
                    "quality_warnings",
                ]
            ].head(
                30
            )
        )


    # ========================================================
    # Style spam
    # ========================================================

    spam = full[
        full[
            "style_spam"
        ]
    ].copy()


    print(
        "\nFull RAG style spam rows:",
        len(
            spam
        ),
    )


    if len(
        spam
    ):

        display(
            spam[
                [
                    "benchmark_uid",
                    "question",
                    "answer",
                ]
            ].head(
                30
            )
        )

Raw result records: 400
Before dedup: 400
After dedup : 400

===== UNIQUE SAMPLE COUNTS =====


,n_unique
variant,
vanilla,100
stage1,100
stage12_weights,100
stage12_full_rag,100


✅ vanilla: 100/100
✅ stage1: 100/100
✅ stage12_weights: 100/100
✅ stage12_full_rag: 100/100

FINAL COMPARISON — Phase 9 v3


,variant,n,semantic_similarity,token_f1,rougeL_f1,year_f1,behavior_accuracy_all,off_topic_refusal_accuracy,history_behavior_accuracy,answer_quality_pass_rate,...,source_validity,unsupported_year_free,retrieval_recall20,retrieval_recall_final,mrr20,grounding_support,context_title_diversity,rewrite_rate,avg_query_variants,avg_latency_sec
0,vanilla,100,0.8627,0.2999,0.2116,0.5041,0.77,0.0,0.8556,0.9667,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.5792
1,stage1,100,0.8301,0.2030,0.1559,0.1948,0.78,0.0,0.8667,0.9000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.5768
2,stage12_weights,100,0.8579,0.3429,0.2797,0.5533,0.82,0.2,0.8889,0.5667,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.3732
3,stage12_full_rag,100,0.8741,0.4332,0.3706,0.6294,0.86,1.0,0.8444,0.9000,...,1.0,0.93,0.7654,0.6173,0.48,0.8727,0.7333,0.03,1.38,1.8036



Saved:
/content/drive/MyDrive/vn_history_model_backups/rag_corpus_vn_history/phase9_hybrid_rag/benchmark_summary_v3_unique_batched.csv

===== FINAL SAMPLE VALIDATION =====
✅ vanilla               : 100/100
✅ stage1                : 100/100
✅ stage12_weights       : 100/100
✅ stage12_full_rag      : 100/100

✅ Benchmark đầy đủ 100 câu × 4 variants.

===== FULL RAG DIAGNOSTICS =====
Full RAG style spam: 1 / 90
Full RAG quality pass: 0.9
Full RAG rewrite rate: 0.03
Mean context diversity: 0.7333
Recall@20: 0.7654
Final context recall: 0.6173
MRR@20: 0.48
Grounding support: 0.8727

Full RAG behavior errors: 14


,benchmark_uid,id,type,question,answer,status,gold_source_ids,source_ids
306,0006_sample_0013_08cd2ef5e252,sample_0013,noisy_context,Trần Hưng Đạo nêu thượng sách giữ nước là gì t...,"Tài liệu được cung cấp nói về cuộc đời, công l...",ok,[hf_wikipedia_trần_hưng_đạo_0008_04f1ff1b0fe1],[]
309,0009_sample_0033_2f888979ceb2,sample_0033,grounded_qa,Thành Hoa Lư có diện tích hơn bao nhiêu hecta?,Tài liệu nêu Cố đô Hoa Lư có diện tích tự nhiê...,ok,[hf_wikipedia_quần_thể_di_tích_cố_đô_hoa_lư_00...,"[hf_wikipedia_hoa_lư_0000_40660bc5e9a4, hf_wik..."
333,0033_sample_0007_644a2f4b4a3f,sample_0007,noisy_context,Chỉ thị “Nhật - Pháp bắn nhau và hành động của...,Tài liệu được cung cấp chủ yếu nói về các chủ ...,ok,[hf_wikipedia_cách_mạng_tháng_tám_0001_b457887...,[]
336,0036_sample_0039_3b8d8a769a15,sample_0039,false_premise,Chúa Nguyễn là người cai trị Đàng Ngoài và phụ...,"Câu hỏi có tiền đề chưa đúng. Theo tài liệu, Đ...",ok,[hf_wikipedia_chúa_nguyễn_0000_e9a6d6674540],[hf_wikipedia_đàng_ngoài_0000_463cae1996fa]
342,0042_sample_0039_9ba23f78f7dd,sample_0039,false_premise,Đường Trường Sơn là tuyến hậu cần do Mỹ xây dự...,"Câu hỏi có tiền đề chưa đúng. Theo tài liệu, Đ...",ok,[hf_wikipedia_đường_trường_sơn_0000_1ae5f18c4c38],[hf_wikipedia_đường_trường_sơn_0000_1ae5f18c4c38]
348,0048_sample_0018_d4241480ded2,sample_0018,insufficient_context,Ngô Quyền có chính sách thuế cụ thể nào sau kh...,"Sau khi lên ngôi, Quang Trung cho thi hành nhữ...",ok,[],[hf_wikipedia_quang_trung_0023_0ce3cfe00d12]
350,0050_sample_0039_cdc4f229fbba,sample_0039,false_premise,Chiến tranh Đông Dương lần thứ nhất bắt đầu nă...,"Câu hỏi có tiền đề chưa đúng. Theo tài liệu, C...",ok,[hf_wikipedia_chiến_tranh_đông_dương_0000_4352...,[hf_wikipedia_chiến_tranh_đông_dương_0000_4352...
351,0051_sample_0002_6bdc7b126254,sample_0002,noisy_context,Võ Nguyên Giáp giữ vai trò gì trong các cuộc c...,Không đủ bằng chứng trong các tài liệu truy xu...,blocked_unsupported_year,[hf_wikipedia_võ_nguyên_giáp_0000_5f1bd468509d],[]
356,0056_sample_0020_20c4c71ec423,sample_0020,false_premise,Nhà Lý chấm dứt vì Lý Công Uẩn nhường ngôi cho...,"Câu hỏi có tiền đề chưa đúng. Theo tài liệu, n...",ok,[hf_wikipedia_nhà_lý_0000_6f87949eb8ba],"[hf_wikipedia_lý_thái_tổ_0000_efb515abc84b, hf..."
382,0082_sample_0039_4c9733b562ca,sample_0039,false_premise,Gia Long là vua Tây Sơn đánh bại Nguyễn Ánh để...,"Câu hỏi có tiền đề chưa đúng. Theo tài liệu, G...",ok,[hf_wikipedia_gia_long_0000_8ccdc77775e8],[hf_wikipedia_gia_long_0000_8ccdc77775e8]



Full RAG quality failures: 9


,benchmark_uid,question,answer,quality_issue_count,quality_warnings
300,0000_sample_0018_be6949a6c0a0,Phùng Hưng đánh phủ đô hộ và Cao Chính Bình th...,"Phùng Hưng đem quân tuần các ấp lân cận, đánh ...",1,[missing_outcome]
304,0004_sample_0007_9a63f6892dfd,Đoàn 559 được thành lập trong bối cảnh nào năm...,Năm 1959 Bộ Chính trị Trung ương Đảng quyết đị...,1,[missing_context]
313,0013_sample_0012_99f2fd369bc6,Cuộc giảng hòa lần thứ nhất giữa nghĩa quân Yê...,"Tháng 10 năm 1894, Pháp chấp nhận giảng hòa, c...",1,[missing_context]
326,0026_sample_0020_3aefe922e0c1,"Trong những năm cuối đời, Phan Bội Châu sống v...","Trong 15 năm cuối đời, Phan Bội Châu vẫn giữ t...",1,[missing_outcome]
328,0028_sample_0009_35b131d036d4,"Thời Lê Nhân Tông, Nguyễn Thái hậu giữ vai trò...","Nguyễn Thị Anh, tức Nguyễn Thần phi, tức Tuyên...",1,[missing_significance]
330,0030_sample_0018_d7ad4d702937,Hiệp ước Versailles năm 1787 quy định những gì...,Hiệp ước Versailles năm 1787 là hiệp ước giữa ...,1,[missing_document_content]
335,0035_sample_0005_a8a1b879e39d,Kiều Công Tiễn có vai trò gì trong bối cảnh dẫ...,"Kiều Công Tiễn là nha tướng Dương Đình Nghệ, t...",1,[missing_significance]
349,0049_sample_0019_1a028175a8da,Dương Đình Nghệ đánh đuổi quân Nam Hán năm 931...,Năm 931 Dương Đình Nghệ tập hợp lực lượng ở Ái...,1,[missing_outcome]
359,0059_sample_0002_0628114daf61,Nhà Lê sơ được thành lập trong hoàn cảnh nào?,"Lê Lợi phát động Khởi nghĩa Lam Sơn, đánh bại ...",1,[missing_context]



Full RAG style spam rows: 1


,benchmark_uid,question,answer
333,0033_sample_0007_644a2f4b4a3f,Chỉ thị “Nhật - Pháp bắn nhau và hành động của...,Tài liệu được cung cấp chủ yếu nói về các chủ ...


## 18. Cách đọc benchmark

Bốn cấu hình được so sánh:

1. **`vanilla`** — Qwen2.5-3B-Instruct nguyên bản, hỏi trực tiếp.
2. **`stage1`** — vanilla + Phase1 LoRA đã merge, hỏi trực tiếp theo prompt Phase1.
3. **`stage12_weights`** — Phase1 + Phase2 đã merge nhưng **không cung cấp retrieval context**. Mục đích là xem Phase2 có học hành vi abstain/grounding hay không; không kỳ vọng cấu hình này có answer-quality cao.
4. **`stage12_full_rag`** — model merge cả hai stage + FAISS + BM25 + RRF + reranker + metadata boost + OOD/source/year guards + LangChain.

Metrics chính:

- `semantic_similarity`, `token_f1`, `rougeL_f1`: độ gần đáp án gold.
- `year_f1`: độ đúng các mốc năm so với gold.
- `behavior_accuracy_all`: trả lời khi nên trả lời và từ chối khi nên từ chối.
- `off_topic_refusal_accuracy`: riêng 10 câu OOD.
- `source_f1`: source IDs model chọn so với gold evidence.
- `source_validity`: model có bịa source ID ngoài context hay không.
- `unsupported_year_free`: không sinh mốc năm ngoài evidence.
- `retrieval_recall20`: gold evidence có nằm trong 20 candidates sau RRF hay không.
- `retrieval_recall_final`: gold evidence có sống sót tới context cuối sau reranker + metadata không.
- `mrr20`: gold evidence đứng cao đến đâu trong 20 candidates.
- `grounding_support`: semantic support proxy giữa answer và chunk được cite.

Không nên chỉ nhìn một metric. Với hệ thống chống bịa, **behavior accuracy + source validity + retrieval recall + answer similarity** quan trọng hơn một điểm tổng hợp duy nhất.
